In [ ]:
'''
$$ \boxed{\text{QEC environment} \rightarrow \text{RL decoder} \rightarrow \text{baseline comparison} \rightarrow \text{scaling}} $$
The agent observes syndrome information and learns a policy for applying corrections.

For a surface code:

$$ \text{noise} \rightarrow \text{syndrome} \rightarrow \boxed{\text{RL agent}} \rightarrow \text{correction} \rightarrow \text{logical success/failure}. $$

We can then compare:
    Majority baseline
    PyMatching
    Supervised MLP
    RL decoder

Using: $$ d=3,\;5,\;7 $$ but not necessarily train everything from scratch independently.

A particularly interesting RL experiment is:

Stage 1 — \(d=3\)

Use \(d=3\) to make sure the RL environment and reward formulation are correct.

Stage 2 — \(d=5\)

Train the RL agent on the larger code.

Stage 3 — \(d=7\)

Test whether the approach continues to work as the code becomes larger.

Stage 4 — generalization

Potentially:

$$ \text{Train at }d=5 \quad\longrightarrow\quad \text{test at }d=7 $$


=============================================================================
State

The current syndrome:

$$ s_t=(s_1,\ldots,s_N) $$

possibly together with information about previous actions.

Action

The agent chooses a correction operation.

For example:

$$ a_t \in \{\text{apply correction at location }i\}. $$

Depending on the representation, an action could correspond to applying an \(X\), \(Z\), or other Pauli correction to a particular data qubit.

Environment

The environment:

samples a physical error;
generates the syndrome;
presents the state to the agent;
accepts correction actions;
evaluates whether the final state is logically correct.
Reward

A simple starting reward could be:

$$ R = \begin{cases} +1 & \text{logical correction succeeds}\\ -1 & \text{logical correction fails}. \end{cases} $$

But I wouldn't immediately use only this reward. QEC rewards can be sparse, making RL unnecessarily difficult.

We can investigate a shaped reward based on syndrome reduction, while ensuring that it doesn't accidentally change the actual decoding objective.

Our current MLP does:

$$ \boxed{s\rightarrow y} $$

where \(y\) is the logical class.

That's supervised classification.

The RL decoder would instead learn:

$$ \boxed{s\rightarrow a_1\rightarrow a_2\rightarrow\cdots\rightarrow\text{correction}} $$

and ultimately try to minimize logical failures.

That is much closer to an active decoder.

'''

In [3]:
'''
PHASE 1
Supervised baseline
        │
        ├── Stim noise model
        ├── MLP decoder
        └── PyMatching baseline
                 │
                 ▼
PHASE 2
RL environment
        │
        ├── State representation
        ├── Action space
        ├── Transition dynamics
        └── Reward
                 │
                 ▼
PHASE 3
RL decoder on d=3
        │
        ├── DQN / PPO
        ├── training stability
        └── comparison with baselines
                 │
                 ▼
PHASE 4
Scaling
        │
        ├── d=5
        └── d=7
                 │
                 ▼
PHASE 5
Generalization
        │
        ├── unseen noise rates
        └── potentially cross-distance
                 │
                 ▼
FINAL
ML vs PyMatching vs RL
'''

'\nPHASE 1\nSupervised baseline\n        │\n        ├── Stim noise model\n        ├── MLP decoder\n        └── PyMatching baseline\n                 │\n                 ▼\nPHASE 2\nRL environment\n        │\n        ├── State representation\n        ├── Action space\n        ├── Transition dynamics\n        └── Reward\n                 │\n                 ▼\nPHASE 3\nRL decoder on d=3\n        │\n        ├── DQN / PPO\n        ├── training stability\n        └── comparison with baselines\n                 │\n                 ▼\nPHASE 4\nScaling\n        │\n        ├── d=5\n        └── d=7\n                 │\n                 ▼\nPHASE 5\nGeneralization\n        │\n        ├── unseen noise rates\n        └── potentially cross-distance\n                 │\n                 ▼\nFINAL\nML vs PyMatching vs RL\n'

In [4]:
# ============================================================
# Reinforcement Learning for Quantum Error-Correction Decoding
# ============================================================

import random
import numpy as np
import stim
import torch

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Initial RL development configuration
# ------------------------------------------------------------

DISTANCE = 3
ROUNDS = 3

NOISE_RATE = 0.01

print("=" * 70)
print("RL QEC DECODER — INITIAL CONFIGURATION")
print("=" * 70)

print(f"Distance       : d = {DISTANCE}")
print(f"Rounds         : {ROUNDS}")
print(f"Noise rate     : p = {NOISE_RATE}")
print(f"Random seed    : {SEED}")
print(f"PyTorch device : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

RL QEC DECODER — INITIAL CONFIGURATION
Distance       : d = 3
Rounds         : 3
Noise rate     : p = 0.01
Random seed    : 42
PyTorch device : cuda
GPU            : NVIDIA GeForce RTX 3050 A Laptop GPU


In [5]:
# ============================================================
# Build the rotated surface-code memory experiment
# ============================================================

def build_surface_code_circuit(
    distance: int,
    rounds: int,
    noise_rate: float,
) -> stim.Circuit:
    """
    Construct the rotated X-memory surface-code circuit
    using the same noise model as the supervised baseline.
    """

    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=noise_rate,
        before_round_data_depolarization=noise_rate,
        before_measure_flip_probability=noise_rate,
        after_reset_flip_probability=noise_rate,
    )

    return circuit


circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=NOISE_RATE,
)

print("=" * 70)
print("SURFACE-CODE CIRCUIT")
print("=" * 70)

print(f"Number of qubits      : {circuit.num_qubits}")
print(f"Number of detectors   : {circuit.num_detectors}")
print(f"Number of observables : {circuit.num_observables}")
print(f"Number of ticks       : {circuit.num_ticks}")

SURFACE-CODE CIRCUIT
Number of qubits      : 26
Number of detectors   : 24
Number of observables : 1
Number of ticks       : 21


In [6]:
# ============================================================
# Inspect the generated Stim circuit
# ============================================================

print("=" * 70)
print("STIM CIRCUIT")
print("=" * 70)

print(circuit)

STIM CIRCUIT
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
RX 1 3 5 8 10 12 15 17 19
Z_ERROR(0.01) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(0.01) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(0.01) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(0.01) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(0.01) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(0.01) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(0.01) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
DEPOLARIZE2(0.01) 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
DEPOLARIZE1(0.01) 2 11 16 25
T

In [7]:
# ============================================================
# Detector Error Model
# ============================================================

dem = circuit.detector_error_model(
    decompose_errors=True
)

print("=" * 70)
print("DETECTOR ERROR MODEL")
print("=" * 70)

print(f"Number of detectors   : {circuit.num_detectors}")
print(f"Number of observables : {circuit.num_observables}")
print()

print(dem)

DETECTOR ERROR MODEL
Number of detectors   : 24
Number of observables : 1

error(0.01911873511075361937) D0 D2
error(0.03501987560309133274) D0 D4
error(0.01911873511075361937) D0 L0
error(0.02169031111111104082) D1 D2
error(0.005333333333333313206) D1 D2 ^ D5
error(0.01911873511075361937) D1 D3
error(0.001338700097173320998) D1 D4
error(0.001338700097173320998) D1 D4 ^ D5
error(0.002006705456917235505) D1 D6
error(0.002006705456917235505) D1 D6 ^ D5
error(0.0006697986788568588423) D1 D6 ^ D5 ^ D8
error(0.0006697986788568588423) D1 D6 ^ D8
error(0.0356427617491228213) D1 D9
error(0.0006697986788568588423) D1 D9 ^ D5
error(0.0006697986788568588423) D1 D9 ^ D5 ^ D8
error(0.0006697986788568588423) D1 D9 ^ D8
error(0.04059590562379881973) D1 L0
error(0.0006697986788568588423) D1 L0 ^ D4 L0
error(0.0006697986788568588423) D1 L0 ^ D4 L0 ^ D5
error(0.0006697986788568588423) D1 L0 ^ D4 L0 ^ D5 ^ D8
error(0.0006697986788568588423) D1 L0 ^ D4 L0 ^ D8
error(0.0006697986788568588423) D1 L0 ^ D5
er

In [ ]:
'''
              hidden physical error
                       │
                       ↓
                Stim experiment
                       │
              ┌────────┴────────┐
              ↓                 ↓
         syndrome D          error E
              │                 │
              ↓                 │
        RL observation          │
              │                 │
              ↓                 │
          D + C₀                │
              │                 │
          ┌───┴────┐            │
          ↓        ↓            │
         Z_i      STOP          │
          │                     │
          ↓                     │
       D + C₁                   │
          │                     │
         ...                    │
          │                     │
          ↓                     │
         STOP                   │
          │                     │
          └──────────┬──────────┘
                     ↓
                 evaluate
                   E × C
                     ↓
             logical success?

'''

In [8]:
# ============================================================
# Data and ancilla qubit structure
# ============================================================

DATA_QUBITS = [
    1, 3, 5,
    8, 10, 12,
    15, 17, 19,
]

ANCILLA_QUBITS = [
    2, 9, 11, 13,
    14, 16, 18, 25,
]

DATA_COORDS = {
    1:  (1, 1),
    3:  (3, 1),
    5:  (5, 1),
    8:  (1, 3),
    10: (3, 3),
    12: (5, 3),
    15: (1, 5),
    17: (3, 5),
    19: (5, 5),
}

ANCILLA_COORDS = {
    2:  (2, 0),
    9:  (2, 2),
    11: (4, 2),
    13: (6, 2),
    14: (0, 4),
    16: (2, 4),
    18: (4, 4),
    25: (4, 6),
}

print("=" * 70)
print("DATA QUBITS")
print("=" * 70)

for q in DATA_QUBITS:
    print(f"q{q:2d} -> coordinate {DATA_COORDS[q]}")

print()

print("=" * 70)
print("ANCILLA QUBITS")
print("=" * 70)

for q in ANCILLA_QUBITS:
    print(f"q{q:2d} -> coordinate {ANCILLA_COORDS[q]}")

print()

print(f"Number of data qubits   : {len(DATA_QUBITS)}")
print(f"Number of ancilla qubits: {len(ANCILLA_QUBITS)}")

DATA QUBITS
q 1 -> coordinate (1, 1)
q 3 -> coordinate (3, 1)
q 5 -> coordinate (5, 1)
q 8 -> coordinate (1, 3)
q10 -> coordinate (3, 3)
q12 -> coordinate (5, 3)
q15 -> coordinate (1, 5)
q17 -> coordinate (3, 5)
q19 -> coordinate (5, 5)

ANCILLA QUBITS
q 2 -> coordinate (2, 0)
q 9 -> coordinate (2, 2)
q11 -> coordinate (4, 2)
q13 -> coordinate (6, 2)
q14 -> coordinate (0, 4)
q16 -> coordinate (2, 4)
q18 -> coordinate (4, 4)
q25 -> coordinate (4, 6)

Number of data qubits   : 9
Number of ancilla qubits: 8


In [9]:
# ============================================================
# Initial RL action space
# ============================================================

ACTION_NAMES = [
    "Z_q1",
    "Z_q3",
    "Z_q5",
    "Z_q8",
    "Z_q10",
    "Z_q12",
    "Z_q15",
    "Z_q17",
    "Z_q19",
    "STOP",
]

NUM_ACTIONS = len(ACTION_NAMES)

print("=" * 70)
print("RL ACTION SPACE")
print("=" * 70)

for action_id, action_name in enumerate(ACTION_NAMES):
    print(f"{action_id:2d} -> {action_name}")

print()
print(f"Number of actions: {NUM_ACTIONS}")

RL ACTION SPACE
 0 -> Z_q1
 1 -> Z_q3
 2 -> Z_q5
 3 -> Z_q8
 4 -> Z_q10
 5 -> Z_q12
 6 -> Z_q15
 7 -> Z_q17
 8 -> Z_q19
 9 -> STOP

Number of actions: 10


In [10]:
# ============================================================
# Correction representation
# ============================================================

NUM_DATA_QUBITS = len(DATA_QUBITS)

correction = np.zeros(NUM_DATA_QUBITS, dtype=np.uint8)

print("=" * 70)
print("INITIAL CORRECTION VECTOR")
print("=" * 70)

print(correction)
print(f"Dimension: {correction.shape}")

INITIAL CORRECTION VECTOR
[0 0 0 0 0 0 0 0 0]
Dimension: (9,)


In [ ]:
'''
| Component            | Initial design                       |
| -------------------- | ------------------------------------ |
| Code                 | Rotated surface code                 |
| Memory               | `rotated_memory_x`                   |
| Development distance | \(d=3\)                              |
| Rounds               | 3                                    |
| Data qubits          | 9                                    |
| Detectors            | 24                                   |
| State                | 24 syndrome bits + 9 correction bits |
| State dimension      | 33                                   |
| Actions              | 9 \(Z_i\) corrections + STOP         |
| Action count         | 10                                   |
| Hidden information   | physical error                       |
| Reward               | terminal logical success/failure     |
| Initial reward       | \(+1/-1\)                            |
| Max steps            | TBD, initially ≤ 9                   |
| Algorithm            | DQN                                  |
| Test set             | untouched                            |

'''

In [11]:
# ============================================================
# CONTROLLED Z-ERROR EXPERIMENT
# ============================================================

print("=" * 70)
print("CONTROLLED Z-ERROR EXPERIMENT")
print("=" * 70)

# Use a completely noise-free circuit.
# This removes stochastic physical noise so that any detector
# changes come only from the deliberately inserted Z error.

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)

print(f"Noise-free circuit:")
print(f"  Qubits      : {clean_circuit.num_qubits}")
print(f"  Detectors   : {clean_circuit.num_detectors}")
print(f"  Observables : {clean_circuit.num_observables}")


def insert_z_after_first_tick(circuit: stim.Circuit, qubit: int) -> stim.Circuit:
    """
    Insert a deterministic Z gate immediately after the first TICK.

    In the generated rotated-memory circuit, this places the
    intentional data-qubit error immediately before the first
    stabilizer-extraction CNOT layer.

    The circuit itself remains otherwise unchanged.
    """

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError("Could not find the first TICK in the circuit.")

    insert_position = position + len(marker)

    modified_text = (
        circuit_text[:insert_position]
        + f"Z {qubit}\n"
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


def sample_syndrome_and_logical(circuit: stim.Circuit):
    """
    Sample one detector record and one logical observable
    from a Stim circuit.
    """

    sampler = circuit.compile_detector_sampler()

    detector_samples, observable_samples = sampler.sample(
        shots=1,
        separate_observables=True,
    )

    detector_record = detector_samples[0].astype(np.uint8)
    logical_record = observable_samples[0].astype(np.uint8)

    return detector_record, logical_record


# ------------------------------------------------------------
# 1. BASELINE: NO INTENTIONAL ERROR
# ------------------------------------------------------------

baseline_detectors, baseline_logical = sample_syndrome_and_logical(
    clean_circuit
)

print("\n" + "-" * 70)
print("BASELINE — NO INTENTIONAL ERROR")
print("-" * 70)

print("Detector syndrome:")
print(baseline_detectors)

print(f"Logical observable L0: {baseline_logical[0]}")

print(
    f"Number of triggered detectors: "
    f"{np.sum(baseline_detectors)}"
)


# ------------------------------------------------------------
# 2. SINGLE-QUBIT Z ERRORS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SINGLE DATA-QUBIT Z ERRORS")
print("-" * 70)

single_error_results = {}

for data_index, qubit in enumerate(DATA_QUBITS):

    error_circuit = insert_z_after_first_tick(
        clean_circuit,
        qubit,
    )

    detector_record, logical_record = sample_syndrome_and_logical(
        error_circuit
    )

    # Compare against the clean baseline.
    changed_detectors = np.flatnonzero(
        detector_record != baseline_detectors
    )

    single_error_results[qubit] = {
        "data_index": data_index,
        "detectors": detector_record,
        "changed_detectors": changed_detectors,
        "logical": int(logical_record[0]),
    }

    print(
        f"\nZ error on q{qubit:2d} "
        f"at coordinate {DATA_COORDS[qubit]}"
    )

    print(
        f"  Changed detectors : "
        f"{changed_detectors.tolist()}"
    )

    print(
        f"  Number changed    : "
        f"{len(changed_detectors)}"
    )

    print(
        f"  Logical L0        : "
        f"{int(logical_record[0])}"
    )


# ------------------------------------------------------------
# 3. COMPACT SYNDROME MAP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA ERROR -> DETECTOR SYNDROME MAP")
print("=" * 70)

for qubit in DATA_QUBITS:

    result = single_error_results[qubit]

    print(
        f"q{qubit:2d} "
        f"{str(DATA_COORDS[qubit]):>8} "
        f"-> "
        f"D{result['changed_detectors'].tolist()} "
        f"| L0={result['logical']}"
    )


# ------------------------------------------------------------
# 4. VERIFY BASELINE DETERMINISM
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASELINE DETERMINISM CHECK")
print("=" * 70)

baseline_sampler = clean_circuit.compile_detector_sampler()

test_samples = baseline_sampler.sample(shots=20)

unique_detector_records = np.unique(
    test_samples,
    axis=0,
)

print(f"Shots tested                 : 20")
print(
    f"Unique detector syndromes   : "
    f"{len(unique_detector_records)}"
)

if len(unique_detector_records) == 1:
    print("PASS: noise-free baseline is deterministic.")
else:
    print("WARNING: baseline is not deterministic.")


# ------------------------------------------------------------
# 5. CHECK WHETHER ANY SINGLE ERROR FLIPS THE LOGICAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SINGLE-ERROR LOGICAL CHECK")
print("=" * 70)

logical_flips = []

for qubit in DATA_QUBITS:

    logical_value = single_error_results[qubit]["logical"]

    if logical_value != int(baseline_logical[0]):
        logical_flips.append(qubit)

print(
    "Single Z errors that change L0:",
    logical_flips
)

if len(logical_flips) == 0:
    print(
        "No single data-qubit Z error flips the logical "
        "observable."
    )
else:
    print(
        "Some single-qubit errors flip the logical observable."
    )

CONTROLLED Z-ERROR EXPERIMENT
Noise-free circuit:
  Qubits      : 26
  Detectors   : 24
  Observables : 1

----------------------------------------------------------------------
BASELINE — NO INTENTIONAL ERROR
----------------------------------------------------------------------
Detector syndrome:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Logical observable L0: 0
Number of triggered detectors: 0

----------------------------------------------------------------------
SINGLE DATA-QUBIT Z ERRORS
----------------------------------------------------------------------

Z error on q 1 at coordinate (1, 1)
  Changed detectors : []
  Number changed    : 0
  Logical L0        : 0

Z error on q 3 at coordinate (3, 1)
  Changed detectors : []
  Number changed    : 0
  Logical L0        : 0

Z error on q 5 at coordinate (5, 1)
  Changed detectors : []
  Number changed    : 0
  Logical L0        : 0

Z error on q 8 at coordinate (1, 3)
  Changed detectors : []
  Number changed    : 0
  Logi

In [12]:
# ============================================================
# DEBUG THE CONTROLLED-ERROR INSERTION
# ============================================================

print("=" * 70)
print("DEBUGGING CONTROLLED Z-ERROR INSERTION")
print("=" * 70)

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)

test_qubit = DATA_QUBITS[0]   # q1

modified_circuit = insert_z_after_first_tick(
    clean_circuit,
    test_qubit,
)

print(f"\nTesting deterministic Z error on q{test_qubit}")
print(f"Coordinate: {DATA_COORDS[test_qubit]}")

print("\n" + "-" * 70)
print("FIRST 25 INSTRUCTIONS OF ORIGINAL CIRCUIT")
print("-" * 70)

original_lines = str(clean_circuit).splitlines()

for i, line in enumerate(original_lines[:25]):
    print(f"{i:02d}: {line}")


print("\n" + "-" * 70)
print("FIRST 25 INSTRUCTIONS OF MODIFIED CIRCUIT")
print("-" * 70)

modified_lines = str(modified_circuit).splitlines()

for i, line in enumerate(modified_lines[:25]):
    print(f"{i:02d}: {line}")


print("\n" + "-" * 70)
print("LOOKING FOR INSERTED Z")
print("-" * 70)

z_locations = [
    (i, line)
    for i, line in enumerate(modified_lines)
    if line.strip() == f"Z {test_qubit}"
]

print(f"Inserted Z locations: {z_locations}")


print("\n" + "-" * 70)
print("CIRCUIT STRUCTURE")
print("-" * 70)

print(f"Original number of qubits     : {clean_circuit.num_qubits}")
print(f"Modified number of qubits     : {modified_circuit.num_qubits}")

print(f"Original detectors            : {clean_circuit.num_detectors}")
print(f"Modified detectors            : {modified_circuit.num_detectors}")

print(f"Original observables          : {clean_circuit.num_observables}")
print(f"Modified observables          : {modified_circuit.num_observables}")


print("\n" + "-" * 70)
print("ORIGINAL CIRCUIT DETECTOR SAMPLING")
print("-" * 70)

original_detectors, original_observable = (
    clean_circuit
    .compile_detector_sampler()
    .sample(
        shots=1,
        separate_observables=True,
    )
)

print("Detectors :", original_detectors[0])
print("L0        :", original_observable[0])


print("\n" + "-" * 70)
print("MODIFIED CIRCUIT DETECTOR SAMPLING")
print("-" * 70)

modified_detectors, modified_observable = (
    modified_circuit
    .compile_detector_sampler()
    .sample(
        shots=1,
        separate_observables=True,
    )
)

print("Detectors :", modified_detectors[0])
print("L0        :", modified_observable[0])


print("\n" + "-" * 70)
print("DIFFERENCE")
print("-" * 70)

changed = np.flatnonzero(
    original_detectors[0] != modified_detectors[0]
)

print("Changed detector indices:", changed.tolist())
print("Number changed          :", len(changed))

print(
    "Logical changed:",
    int(original_observable[0, 0])
    != int(modified_observable[0, 0])
)

DEBUGGING CONTROLLED Z-ERROR INSERTION

Testing deterministic Z error on q1
Coordinate: (1, 1)

----------------------------------------------------------------------
FIRST 25 INSTRUCTIONS OF ORIGINAL CIRCUIT
----------------------------------------------------------------------
00: QUBIT_COORDS(1, 1) 1
01: QUBIT_COORDS(2, 0) 2
02: QUBIT_COORDS(3, 1) 3
03: QUBIT_COORDS(5, 1) 5
04: QUBIT_COORDS(1, 3) 8
05: QUBIT_COORDS(2, 2) 9
06: QUBIT_COORDS(3, 3) 10
07: QUBIT_COORDS(4, 2) 11
08: QUBIT_COORDS(5, 3) 12
09: QUBIT_COORDS(6, 2) 13
10: QUBIT_COORDS(0, 4) 14
11: QUBIT_COORDS(1, 5) 15
12: QUBIT_COORDS(2, 4) 16
13: QUBIT_COORDS(3, 5) 17
14: QUBIT_COORDS(4, 4) 18
15: QUBIT_COORDS(5, 5) 19
16: QUBIT_COORDS(4, 6) 25
17: RX 1 3 5 8 10 12 15 17 19
18: R 2 9 11 13 14 16 18 25
19: TICK
20: H 2 11 16 25
21: TICK
22: CX 2 3 16 17 11 12 15 14 10 9 19 18
23: TICK
24: CX 2 1 16 15 11 10 8 14 3 9 12 18

----------------------------------------------------------------------
FIRST 25 INSTRUCTIONS OF MODIFIE

In [13]:
# ============================================================
# BASIC STIM PAULI SANITY CHECK
# ============================================================

print("=" * 70)
print("BASIC STIM PAULI SANITY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# TEST 1: |+> WITHOUT Z -> X measurement should be 0
# ------------------------------------------------------------

circuit_plus = stim.Circuit("""
RX 0
MX 0
""")

sample_plus = circuit_plus.compile_sampler().sample(
    shots=20
)

print("\nTEST 1: |+> -> MX")
print("Samples:")
print(sample_plus.flatten())

print(
    "Unique results:",
    np.unique(sample_plus)
)


# ------------------------------------------------------------
# TEST 2: |+> -> Z -> MX should give 1
# ------------------------------------------------------------

circuit_minus = stim.Circuit("""
RX 0
Z 0
MX 0
""")

sample_minus = circuit_minus.compile_sampler().sample(
    shots=20
)

print("\nTEST 2: |+> -> Z -> MX")
print("Samples:")
print(sample_minus.flatten())

print(
    "Unique results:",
    np.unique(sample_minus)
)


# ------------------------------------------------------------
# TEST 3: SIMPLE X-STABILIZER MEASUREMENT
#
# Ancilla starts in |0>
# H -> |+>
# CNOT ancilla -> data
# H
# M
#
# A data Z error should flip the ancilla result.
# ------------------------------------------------------------

simple_no_error = stim.Circuit("""
RX 0
R 1
H 1
CX 1 0
H 1
M 1
""")

simple_z_error = stim.Circuit("""
RX 0
R 1
Z 0
H 1
CX 1 0
H 1
M 1
""")


no_error_samples = (
    simple_no_error
    .compile_sampler()
    .sample(shots=20)
    .flatten()
)

z_error_samples = (
    simple_z_error
    .compile_sampler()
    .sample(shots=20)
    .flatten()
)

print("\nTEST 3: SIMPLE X-STABILIZER MEASUREMENT")

print("Without Z error:")
print(no_error_samples)

print("With Z error:")
print(z_error_samples)

print(
    "No-error unique result:",
    np.unique(no_error_samples)
)

print(
    "Z-error unique result:",
    np.unique(z_error_samples)
)


# ------------------------------------------------------------
# TEST 4: SAME SIMPLE CIRCUIT WITH A DETECTOR
# ------------------------------------------------------------

simple_detector_no_error = stim.Circuit("""
RX 0
R 1
H 1
CX 1 0
H 1
M 1
DETECTOR rec[-1]
""")

simple_detector_z_error = stim.Circuit("""
RX 0
R 1
Z 0
H 1
CX 1 0
H 1
M 1
DETECTOR rec[-1]
""")


det_no_error = (
    simple_detector_no_error
    .compile_detector_sampler()
    .sample(shots=1)
)

det_z_error = (
    simple_detector_z_error
    .compile_detector_sampler()
    .sample(shots=1)
)

print("\nTEST 4: DETECTOR RESPONSE")

print(
    "Detector without Z:",
    det_no_error[0]
)

print(
    "Detector with Z:",
    det_z_error[0]
)

BASIC STIM PAULI SANITY CHECK

TEST 1: |+> -> MX
Samples:
[False False False False False False False False False False False False
 False False False False False False False False]
Unique results: [False]

TEST 2: |+> -> Z -> MX
Samples:
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True]
Unique results: [ True]

TEST 3: SIMPLE X-STABILIZER MEASUREMENT
Without Z error:
[False False False False False False False False False False False False
 False False False False False False False False]
With Z error:
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True]
No-error unique result: [False]
Z-error unique result: [ True]

TEST 4: DETECTOR RESPONSE
Detector without Z: [False]
Detector with Z: [False]


In [14]:
# ============================================================
# CONTROLLED Z ERROR — RAW MEASUREMENT COMPARISON
# ============================================================

print("=" * 70)
print("CONTROLLED Z ERROR — RAW MEASUREMENT COMPARISON")
print("=" * 70)


# ------------------------------------------------------------
# BUILD NOISE-FREE CIRCUITS
# ------------------------------------------------------------

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)


def insert_z_after_first_tick(circuit: stim.Circuit, qubit: int) -> stim.Circuit:
    """
    Insert a deterministic Z gate immediately after the first TICK.

    This places the error before the first stabilizer-extraction
    CNOT layer.
    """

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError(
            "Could not find the first TICK in the circuit."
        )

    insert_position = position + len(marker)

    modified_text = (
        circuit_text[:insert_position]
        + f"Z {qubit}\n"
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


# ------------------------------------------------------------
# RAW MEASUREMENT SAMPLER
# ------------------------------------------------------------

def sample_raw_measurements(circuit: stim.Circuit):
    """
    Return the raw measurement record from a circuit.
    """

    sampler = circuit.compile_sampler()

    sample = sampler.sample(
        shots=1
    )

    return sample[0].astype(np.uint8)


# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

baseline_measurements = sample_raw_measurements(
    clean_circuit
)

print("\n" + "-" * 70)
print("BASELINE")
print("-" * 70)

print(
    f"Total measurements: "
    f"{len(baseline_measurements)}"
)

print("Raw measurement record:")
print(baseline_measurements)


# ------------------------------------------------------------
# TEST EVERY DATA QUBIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SINGLE DATA-QUBIT Z ERRORS")
print("=" * 70)

raw_error_results = {}

for qubit in DATA_QUBITS:

    error_circuit = insert_z_after_first_tick(
        clean_circuit,
        qubit,
    )

    error_measurements = sample_raw_measurements(
        error_circuit
    )

    changed_measurements = np.flatnonzero(
        error_measurements != baseline_measurements
    )

    raw_error_results[qubit] = {
        "measurements": error_measurements,
        "changed_measurements": changed_measurements,
    }

    print(
        f"\nZ error on q{qubit:2d} "
        f"at coordinate {DATA_COORDS[qubit]}"
    )

    print(
        f"  Changed measurements : "
        f"{changed_measurements.tolist()}"
    )

    print(
        f"  Number changed       : "
        f"{len(changed_measurements)}"
    )


# ------------------------------------------------------------
# COMPACT RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RAW MEASUREMENT ERROR MAP")
print("=" * 70)

for qubit in DATA_QUBITS:

    changed = raw_error_results[qubit][
        "changed_measurements"
    ]

    print(
        f"q{qubit:2d} "
        f"{str(DATA_COORDS[qubit]):>8} "
        f"-> measurements {changed.tolist()}"
    )


# ------------------------------------------------------------
# IMPORTANT: CHECK TOTAL MEASUREMENT COUNT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEASUREMENT STRUCTURE")
print("=" * 70)

print(
    f"Total raw measurements: "
    f"{len(baseline_measurements)}"
)

print(
    "Expected final data measurements: "
    f"{NUM_DATA_QUBITS}"
)

print(
    "Expected ancilla measurements: "
    f"{ROUNDS * len(ANCILLA_QUBITS)}"
)

print(
    "Total expected: "
    f"{NUM_DATA_QUBITS + ROUNDS * len(ANCILLA_QUBITS)}"
)

CONTROLLED Z ERROR — RAW MEASUREMENT COMPARISON

----------------------------------------------------------------------
BASELINE
----------------------------------------------------------------------
Total measurements: 33
Raw measurement record:
[0 1 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 1 0 1 1 1 1 1 1 0 0 0]

SINGLE DATA-QUBIT Z ERRORS

Z error on q 1 at coordinate (1, 1)
  Changed measurements : [0, 1, 3, 6, 8, 9, 11, 14, 16, 17, 19, 22, 25, 28, 30]
  Number changed       : 15

Z error on q 3 at coordinate (3, 1)
  Changed measurements : [0, 2, 3, 8, 10, 11, 16, 18, 19, 24, 26, 30, 31, 32]
  Number changed       : 14

Z error on q 5 at coordinate (5, 1)
  Changed measurements : [2, 3, 4, 10, 11, 12, 18, 19, 20, 27, 29, 30]
  Number changed       : 12

Z error on q 8 at coordinate (1, 3)
  Changed measurements : [4, 5, 6, 12, 13, 14, 20, 21, 22, 26, 28, 30, 31, 32]
  Number changed       : 14

Z error on q10 at coordinate (3, 3)
  Changed measurements : [1, 2, 4, 5, 9, 10, 12, 13,

In [ ]:
'''
                 Hidden physical error
                         │
                         ▼
                  Stim QEC circuit
                         │
                         ▼
                 raw measurements
                         │
                         ▼
                  detector events D
                         │
                         ▼
                 ┌───────────────┐
                 │   RL Agent    │
                 │ sees D only   │
                 └───────────────┘
                         │
                         ▼
                 Z correction(s)
                         │
                         ▼
                  logical outcome
'''

In [15]:
# ============================================================
# EXTRACT EXACT DETECTOR DEFINITIONS
# ============================================================

print("=" * 70)
print("EXACT DETECTOR DEFINITIONS")
print("=" * 70)

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)

print("\nDetector error model summary:")
print(f"  Detectors   : {clean_circuit.num_detectors}")
print(f"  Observables : {clean_circuit.num_observables}")
print(f"  Measurements: {clean_circuit.num_measurements}")


# ------------------------------------------------------------
# PRINT CIRCUIT LINES CONTAINING DETECTOR DEFINITIONS
# ------------------------------------------------------------

circuit_lines = str(clean_circuit).splitlines()

detector_lines = [
    line.strip()
    for line in circuit_lines
    if line.strip().startswith("DETECTOR")
]

print("\n" + "-" * 70)
print("DETECTOR INSTRUCTIONS")
print("-" * 70)

for i, line in enumerate(detector_lines):
    print(f"D{i:02d}: {line}")

EXACT DETECTOR DEFINITIONS

Detector error model summary:
  Detectors   : 24
  Observables : 1
  Measurements: 33

----------------------------------------------------------------------
DETECTOR INSTRUCTIONS
----------------------------------------------------------------------
D00: DETECTOR(2, 0, 0) rec[-8]
D01: DETECTOR(2, 4, 0) rec[-3]
D02: DETECTOR(4, 2, 0) rec[-6]
D03: DETECTOR(4, 6, 0) rec[-1]
D04: DETECTOR(2, 0, 0) rec[-8] rec[-16]
D05: DETECTOR(2, 2, 0) rec[-7] rec[-15]
D06: DETECTOR(4, 2, 0) rec[-6] rec[-14]
D07: DETECTOR(6, 2, 0) rec[-5] rec[-13]
D08: DETECTOR(0, 4, 0) rec[-4] rec[-12]
D09: DETECTOR(2, 4, 0) rec[-3] rec[-11]
D10: DETECTOR(4, 4, 0) rec[-2] rec[-10]
D11: DETECTOR(4, 6, 0) rec[-1] rec[-9]
D12: DETECTOR(2, 0, 1) rec[-8] rec[-9] rec[-17]
D13: DETECTOR(2, 4, 1) rec[-2] rec[-3] rec[-5] rec[-6] rec[-12]
D14: DETECTOR(4, 2, 1) rec[-4] rec[-5] rec[-7] rec[-8] rec[-15]
D15: DETECTOR(4, 6, 1) rec[-1] rec[-2] rec[-10]


In [16]:
# ============================================================
# INSPECT STIM CIRCUIT INSTRUCTION TYPES
# ============================================================

print("=" * 70)
print("STIM CIRCUIT INSTRUCTION INSPECTION")
print("=" * 70)

for index, instruction in enumerate(clean_circuit):
    
    name = instruction.name

    if name in {"M", "MR", "MX", "DETECTOR", "OBSERVABLE_INCLUDE"}:
        print(
            f"{index:03d}: "
            f"{name:20s} "
            f"{instruction}"
        )

STIM CIRCUIT INSTRUCTION INSPECTION
032: MR                   MR 2 9 11 13 14 16 18 25
033: DETECTOR             DETECTOR(2, 0, 0) rec[-8]
034: DETECTOR             DETECTOR(2, 4, 0) rec[-3]
035: DETECTOR             DETECTOR(4, 2, 0) rec[-6]
036: DETECTOR             DETECTOR(4, 6, 0) rec[-1]
038: MX                   MX 1 3 5 8 10 12 15 17 19
039: DETECTOR             DETECTOR(2, 0, 1) rec[-8] rec[-9] rec[-17]
040: DETECTOR             DETECTOR(2, 4, 1) rec[-2] rec[-3] rec[-5] rec[-6] rec[-12]
041: DETECTOR             DETECTOR(4, 2, 1) rec[-4] rec[-5] rec[-7] rec[-8] rec[-15]
042: DETECTOR             DETECTOR(4, 6, 1) rec[-1] rec[-2] rec[-10]
043: OBSERVABLE_INCLUDE   OBSERVABLE_INCLUDE(0) rec[-3] rec[-6] rec[-9]


In [17]:
# ============================================================
# PAULI CANCELLATION TEST
# ============================================================

print("=" * 70)
print("PAULI CANCELLATION TEST")
print("=" * 70)


def insert_two_z_after_first_tick(
    circuit: stim.Circuit,
    qubit: int,
) -> stim.Circuit:
    """
    Insert Z_q twice at the same point.

    Since Z^2 = I, this should be physically equivalent
    to inserting no error.
    """

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError(
            "Could not find the first TICK."
        )

    insert_position = position + len(marker)

    modified_text = (
        circuit_text[:insert_position]
        + f"Z {qubit}\n"
        + f"Z {qubit}\n"
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


# ------------------------------------------------------------
# USE q1 AS TEST
# ------------------------------------------------------------

test_qubit = DATA_QUBITS[0]

single_z_circuit = insert_z_after_first_tick(
    clean_circuit,
    test_qubit,
)

double_z_circuit = insert_two_z_after_first_tick(
    clean_circuit,
    test_qubit,
)


clean_raw = sample_raw_measurements(
    clean_circuit
)

single_z_raw = sample_raw_measurements(
    single_z_circuit
)

double_z_raw = sample_raw_measurements(
    double_z_circuit
)


# ------------------------------------------------------------
# COMPARE
# ------------------------------------------------------------

single_difference = np.flatnonzero(
    clean_raw != single_z_raw
)

double_difference = np.flatnonzero(
    clean_raw != double_z_raw
)


print(f"\nTest qubit: q{test_qubit}")
print(f"Coordinate: {DATA_COORDS[test_qubit]}")

print("\nClean vs single Z:")
print(
    "Changed measurements:",
    single_difference.tolist()
)

print(
    "Number changed:",
    len(single_difference)
)

print("\nClean vs double Z:")
print(
    "Changed measurements:",
    double_difference.tolist()
)

print(
    "Number changed:",
    len(double_difference)
)


# ------------------------------------------------------------
# FINAL PASS/FAIL
# ------------------------------------------------------------

print("\n" + "-" * 70)

if len(double_difference) == 0:
    print("PASS: Z^2 = I verified experimentally.")
else:
    print(
        "WARNING: double Z did not cancel as expected."
    )

PAULI CANCELLATION TEST

Test qubit: q1
Coordinate: (1, 1)

Clean vs single Z:
Changed measurements: [0, 1, 6, 8, 9, 14, 16, 17, 22, 24, 27, 30]
Number changed: 12

Clean vs double Z:
Changed measurements: [4, 6, 12, 14, 20, 22, 24, 25, 26, 27, 31, 32]
Number changed: 12

----------------------------------------------------------------------


In [18]:
# ============================================================
# CONTROLLED Z-ERROR EXPERIMENT — CORRECTED
#
# IMPORTANT:
# We do NOT compare independent raw measurement shots.
# Instead, we characterize each circuit statistically and
# examine detector/logical distributions.
# ============================================================

print("=" * 70)
print("CONTROLLED Z-ERROR EXPERIMENT — CORRECTED")
print("=" * 70)


# ------------------------------------------------------------
# NOISE-FREE BASELINE
# ------------------------------------------------------------

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)


def insert_z_after_first_tick(
    circuit: stim.Circuit,
    qubit: int,
) -> stim.Circuit:

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError(
            "Could not find first TICK."
        )

    insert_position = position + len(marker)

    modified_text = (
        circuit_text[:insert_position]
        + f"Z {qubit}\n"
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


def insert_two_z_after_first_tick(
    circuit: stim.Circuit,
    qubit: int,
) -> stim.Circuit:

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError(
            "Could not find first TICK."
        )

    insert_position = position + len(marker)

    modified_text = (
        circuit_text[:insert_position]
        + f"Z {qubit}\n"
        + f"Z {qubit}\n"
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


# ------------------------------------------------------------
# SAMPLE DETECTOR DISTRIBUTION
# ------------------------------------------------------------

def characterize_circuit(
    circuit: stim.Circuit,
    shots: int = 1000,
):
    """
    Characterize detector and logical behavior over many shots.
    """

    sampler = circuit.compile_detector_sampler()

    detector_samples, observable_samples = sampler.sample(
        shots=shots,
        separate_observables=True,
    )

    detector_samples = detector_samples.astype(np.uint8)
    observable_samples = observable_samples.astype(np.uint8)

    detector_counts = np.sum(
        detector_samples,
        axis=0,
    )

    logical_counts = np.sum(
        observable_samples,
        axis=0,
    )

    return (
        detector_samples,
        observable_samples,
        detector_counts,
        logical_counts,
    )


# ------------------------------------------------------------
# BUILD THREE CIRCUITS
# ------------------------------------------------------------

test_qubit = DATA_QUBITS[0]

single_z_circuit = insert_z_after_first_tick(
    clean_circuit,
    test_qubit,
)

double_z_circuit = insert_two_z_after_first_tick(
    clean_circuit,
    test_qubit,
)


# ------------------------------------------------------------
# CHARACTERIZE
# ------------------------------------------------------------

SHOTS = 1000

(
    clean_det,
    clean_obs,
    clean_det_counts,
    clean_obs_counts,
) = characterize_circuit(
    clean_circuit,
    shots=SHOTS,
)

(
    single_det,
    single_obs,
    single_det_counts,
    single_obs_counts,
) = characterize_circuit(
    single_z_circuit,
    shots=SHOTS,
)

(
    double_det,
    double_obs,
    double_det_counts,
    double_obs_counts,
) = characterize_circuit(
    double_z_circuit,
    shots=SHOTS,
)


# ------------------------------------------------------------
# DISPLAY DETECTOR STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DETECTOR STATISTICS")
print("-" * 70)

print(f"Shots: {SHOTS}")
print(f"Test qubit: q{test_qubit}")
print(f"Coordinate: {DATA_COORDS[test_qubit]}")

print("\nDetector firing counts:")
print(
    "D index : clean | single-Z | double-Z"
)

for d in range(clean_circuit.num_detectors):

    print(
        f"D{d:02d}     : "
        f"{clean_det_counts[d]:5d} | "
        f"{single_det_counts[d]:8d} | "
        f"{double_det_counts[d]:8d}"
    )


# ------------------------------------------------------------
# DETECTOR PROBABILITIES
# ------------------------------------------------------------

clean_det_prob = clean_det_counts / SHOTS
single_det_prob = single_det_counts / SHOTS
double_det_prob = double_det_counts / SHOTS


print("\n" + "-" * 70)
print("DETECTOR PROBABILITIES")
print("-" * 70)

print(
    "D index : clean | single-Z | double-Z"
)

for d in range(clean_circuit.num_detectors):

    print(
        f"D{d:02d}     : "
        f"{clean_det_prob[d]:7.3f} | "
        f"{single_det_prob[d]:8.3f} | "
        f"{double_det_prob[d]:8.3f}"
    )


# ------------------------------------------------------------
# LOGICAL OBSERVABLE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOGICAL OBSERVABLE")
print("-" * 70)

print(
    f"Clean logical-1 frequency : "
    f"{clean_obs_counts[0] / SHOTS:.4f}"
)

print(
    f"Single-Z logical-1 freq   : "
    f"{single_obs_counts[0] / SHOTS:.4f}"
)

print(
    f"Double-Z logical-1 freq   : "
    f"{double_obs_counts[0] / SHOTS:.4f}"
)


# ------------------------------------------------------------
# CHECK WHETHER DOUBLE Z HAS SAME DETECTOR DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Z² = I STATISTICAL CHECK")
print("=" * 70)

max_detector_difference = np.max(
    np.abs(
        clean_det_prob
        - double_det_prob
    )
)

logical_difference = abs(
    (clean_obs_counts[0] / SHOTS)
    - (double_obs_counts[0] / SHOTS)
)

print(
    f"Maximum detector probability difference: "
    f"{max_detector_difference:.6f}"
)

print(
    f"Logical probability difference: "
    f"{logical_difference:.6f}"
)

if max_detector_difference < 1e-12 and logical_difference < 1e-12:
    print(
        "PASS: double Z is statistically identical "
        "to the clean circuit."
    )
else:
    print(
        "WARNING: clean and double-Z distributions differ."
    )

CONTROLLED Z-ERROR EXPERIMENT — CORRECTED

----------------------------------------------------------------------
DETECTOR STATISTICS
----------------------------------------------------------------------
Shots: 1000
Test qubit: q1
Coordinate: (1, 1)

Detector firing counts:
D index : clean | single-Z | double-Z
D00     :     0 |        0 |        0
D01     :     0 |        0 |        0
D02     :     0 |        0 |        0
D03     :     0 |        0 |        0
D04     :     0 |        0 |        0
D05     :     0 |        0 |        0
D06     :     0 |        0 |        0
D07     :     0 |        0 |        0
D08     :     0 |        0 |        0
D09     :     0 |        0 |        0
D10     :     0 |        0 |        0
D11     :     0 |        0 |        0
D12     :     0 |        0 |        0
D13     :     0 |        0 |        0
D14     :     0 |        0 |        0
D15     :     0 |        0 |        0
D16     :     0 |        0 |        0
D17     :     0 |        0 |        0
D1

In [ ]:
'''
Stim's measurement-to-detector conversion uses a noiseless reference sample from the circuit.

When we created a new circuit containing the deterministic Z, we also created a new detector sampler whose reference corresponds to that modified circuit. Consequently, the deterministic fault can disappear from the reported detection events.

So the all-zero detector result does not mean the \(Z\) fault is invisible.

Raw-measurement results already proved that the fault changes the stabilizer measurements.

The correct method

We should:

Keep the clean circuit as the reference circuit.
Generate raw measurements from the fault-inserted circuit.
Feed those raw measurements into the clean circuit's compile_m2d_converter().
This converts them into detector events relative to the clean/no-error reference. Stim explicitly provides this measurement-to-detection-events converter for this purpose.

'''

In [19]:
# ============================================================
# CONTROLLED Z ERROR -> DETECTOR EVENTS
#
# IMPORTANT:
# The CLEAN circuit provides the detector reference.
# Fault circuits only provide raw measurement records.
# ============================================================

print("=" * 70)
print("CONTROLLED Z ERROR -> DETECTOR EVENTS")
print("=" * 70)


# ------------------------------------------------------------
# 1. CLEAN REFERENCE CIRCUIT
# ------------------------------------------------------------

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)

print("\nClean reference circuit:")
print(f"  Measurements : {clean_circuit.num_measurements}")
print(f"  Detectors    : {clean_circuit.num_detectors}")
print(f"  Observables  : {clean_circuit.num_observables}")


# ------------------------------------------------------------
# 2. INSERT A DETERMINISTIC Z ERROR
# ------------------------------------------------------------

def insert_z_after_first_tick(
    circuit: stim.Circuit,
    qubit: int,
) -> stim.Circuit:

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError(
            "Could not find the first TICK."
        )

    insert_position = position + len(marker)

    modified_text = (
        circuit_text[:insert_position]
        + f"Z {qubit}\n"
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


# ------------------------------------------------------------
# 3. CLEAN-CIRCUIT MEASUREMENT -> DETECTOR CONVERTER
# ------------------------------------------------------------

converter = clean_circuit.compile_m2d_converter()

print("\nCreated measurement-to-detector converter")
print("Reference: CLEAN / NO-ERROR circuit")


# ------------------------------------------------------------
# 4. SAMPLE RAW MEASUREMENTS FROM A CIRCUIT
# ------------------------------------------------------------

def sample_raw_measurements(
    circuit: stim.Circuit,
    shots: int = 1,
):
    """
    Sample actual measurement outcomes from a circuit.
    """

    return circuit.compile_sampler().sample(
        shots=shots
    )


# ------------------------------------------------------------
# 5. CONVERT RAW MEASUREMENTS USING CLEAN REFERENCE
# ------------------------------------------------------------

def measurements_to_detectors(
    raw_measurements: np.ndarray,
):
    """
    Convert raw measurement records into detector events
    using the clean circuit's reference frame.
    """

    detectors, observables = converter.convert(
        measurements=raw_measurements,
        separate_observables=True,
    )

    return (
        detectors.astype(np.uint8),
        observables.astype(np.uint8),
    )


# ------------------------------------------------------------
# 6. CLEAN BASELINE
# ------------------------------------------------------------

baseline_raw = sample_raw_measurements(
    clean_circuit,
    shots=1,
)

baseline_detectors, baseline_observable = (
    measurements_to_detectors(
        baseline_raw
    )
)

print("\n" + "-" * 70)
print("CLEAN BASELINE")
print("-" * 70)

print("Raw measurements:")
print(baseline_raw[0].astype(np.uint8))

print("\nDetector events:")
print(baseline_detectors[0])

print(
    "\nLogical observable:",
    baseline_observable[0]
)


# ------------------------------------------------------------
# 7. TEST EVERY DATA QUBIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SINGLE DATA-QUBIT Z ERRORS")
print("=" * 70)

controlled_detector_results = {}

for qubit in DATA_QUBITS:

    fault_circuit = insert_z_after_first_tick(
        clean_circuit,
        qubit,
    )

    raw = sample_raw_measurements(
        fault_circuit,
        shots=1,
    )

    detectors, observables = measurements_to_detectors(
        raw
    )

    detector_record = detectors[0]
    observable_record = observables[0]

    changed_detectors = np.flatnonzero(
        detector_record != baseline_detectors[0]
    )

    controlled_detector_results[qubit] = {
        "raw": raw[0],
        "detectors": detector_record,
        "changed_detectors": changed_detectors,
        "observable": int(observable_record[0]),
    }

    print(
        f"\nZ error on q{qubit:2d} "
        f"at coordinate {DATA_COORDS[qubit]}"
    )

    print(
        "  Changed detectors :",
        changed_detectors.tolist()
    )

    print(
        "  Number changed    :",
        len(changed_detectors)
    )

    print(
        "  Logical L0        :",
        int(observable_record[0])
    )


# ------------------------------------------------------------
# 8. COMPACT ERROR -> DETECTOR MAP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHYSICAL Z ERROR -> DETECTOR MAP")
print("=" * 70)

for qubit in DATA_QUBITS:

    result = controlled_detector_results[qubit]

    print(
        f"q{qubit:2d} "
        f"{str(DATA_COORDS[qubit]):>8} "
        f"-> "
        f"D{result['changed_detectors'].tolist()} "
        f"| L0={result['observable']}"
    )


# ------------------------------------------------------------
# 9. CHECK THAT THE CLEAN REFERENCE IS ZERO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REFERENCE CHECK")
print("=" * 70)

if np.sum(baseline_detectors[0]) == 0:
    print(
        "PASS: clean reference produces zero detection events."
    )
else:
    print(
        "WARNING: clean reference produced nonzero "
        "detection events."
    )

CONTROLLED Z ERROR -> DETECTOR EVENTS

Clean reference circuit:
  Measurements : 33
  Detectors    : 24
  Observables  : 1

Created measurement-to-detector converter
Reference: CLEAN / NO-ERROR circuit

----------------------------------------------------------------------
CLEAN BASELINE
----------------------------------------------------------------------
Raw measurements:
[0 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1 1 1 1 1]

Detector events:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Logical observable: [0]

SINGLE DATA-QUBIT Z ERRORS

Z error on q 1 at coordinate (1, 1)
  Changed detectors : [0]
  Number changed    : 1
  Logical L0        : 1

Z error on q 3 at coordinate (3, 1)
  Changed detectors : [0, 2]
  Number changed    : 2
  Logical L0        : 0

Z error on q 5 at coordinate (5, 1)
  Changed detectors : [2]
  Number changed    : 1
  Logical L0        : 0

Z error on q 8 at coordinate (1, 3)
  Changed detectors : [1]
  Number changed    : 1
  Logical 

In [ ]:
'''
the earlier conclusion that:

"single \(Z\) errors produce no detector events."

That conclusion was incorrect.

Likewise, the earlier Z² raw-measurement comparison was not a valid test of \(Z^2=I\), because independently sampled raw measurement records can differ even when they represent the same physical situation.
'''

'''
                    CLEAN CIRCUIT
                         │
                         ▼
                 reference sample
                         │
                         ▼
raw measurements ──► m2d converter ──► detector events

'''

In [ ]:
'''
        So the detector structure is:
                 D0 -------- D2
                  |           |
                q1/q3       q5/q12
                  |           |
                q8/q10 ---- ...
                  |
                 D1 -------- D3

The experimentally verified incidence relationships are:
q1   -> D0
q3   -> D0 ⊕ D2
q5   -> D2

q8   -> D1
q10  -> D1 ⊕ D2
q12  -> D2

q15  -> D1
q17  -> D1 ⊕ D3
q19  -> D3


We can now confidently say that a physical \(Z\) error produces a deterministic detector signature in this noise-free controlled experiment.

For example: $$ Z_{q1}\rightarrow D_0 $$

while $$ Z_{q3}\rightarrow D_0\oplus D_2. $$

Therefore, the decoder's task can genuinely be formulated as:

$$ \text{observed syndrome} \longrightarrow \text{sequence of physical corrections}. $$

---------------------------------

Notice that D0–D3 are the only detectors changing in this experiment.

That does not mean the surface code only has four useful detectors.

It means our injected \(Z\) error was placed after the first TICK, and therefore it affects the relevant subsequent syndrome evolution in a particular way.

The remaining detectors correspond to later spacetime syndrome information and don't necessarily change for this particular fault insertion point.

This is actually useful for us: it demonstrates why the RL environment should eventually treat the syndrome as a spacetime sequence, rather than assuming the 24 detector bits are simply nine static spatial checks.
'''


In [ ]:
'''

We have:

q1  -> L0 = 1
q8  -> L0 = 1
q15 -> L0 = 1

while the other six single-qubit \(Z\) errors give:

L0 = 0

This is consistent with the fact that the three qubits

$$ q_1,\ q_8,\ q_{15} $$

form one vertical logical-\(Z\) string for the rotated memory-\(X\) code.

So a single \(Z\) on one of those logical-string locations flips the logical observable.

However, don't use \(L_0\) as part of the RL observation.

The RL agent must not be given the answer it is supposed to learn.

We can use \(L_0\) internally for evaluation/reward.

'''

'''

We have now validated:
        physical Z fault
               ↓
        Stim circuit
               ↓
        raw measurements
               ↓
        clean reference converter
               ↓
        detector events

But we still need to validate the most important part:
    physical error + RL correction
              ↓
        residual error
              ↓
       logical success?

In other words, we need to make sure that when the environment says:
        hidden error = q3
    agent correction = q3

the resulting state is logically corrected.

And similarly:
    hidden error = q3
    agent correction = q1

must behave differently.

This is the reward-validation experiment.

'''

In [ ]:
'''

Represent the physical \(Z\) error as a 9-bit vector:

[q1 q3 q5 q8 q10 q12 q15 q17 q19]

For example:

q3 error:

[0 1 0 0 0 0 0 0 0]
Correction

Represent the agent's correction in exactly the same way:

q3 correction:

[0 1 0 0 0 0 0 0 0]
Residual error

Because

$$ Z^2=I, $$

the residual is XOR:

$$ E_{\mathrm{residual}} = E_{\mathrm{physical}} \oplus E_{\mathrm{correction}}. $$

Therefore:

physical : 0 1 0 0 0 0 0 0 0
correction:0 1 0 0 0 0 0 0 0
--------------------------------
residual : 0 0 0 0 0 0 0 0 0

Perfect.

But:

physical : 0 1 0 0 0 0 0 0 0
correction:1 0 0 0 0 0 0 0 0
--------------------------------
residual : 1 1 0 0 0 0 0 0 0

is not the same physical error and may produce a logical failure.

'''

'''
There is one more validation we should do

We should test all \(2^9=512\) possible Z-error patterns.

That is tiny.

For every pattern:

$$ E\in\{0,1\}^9 $$

we can:

    construct the corresponding physical \(Z\) faults,
    run the noiseless circuit,
    convert using the clean reference,
    record the 24-bit detector syndrome,
    record \(L_0\).

This gives us a complete lookup table:

    physical Z error
            ↓
    24-bit detector syndrome
            ↓
    logical observable

Then we can test:

whether different physical errors produce identical syndromes,
which errors are logically equivalent,
whether the syndrome contains enough information to choose a correction,
how many correction choices are valid,
whether our proposed RL state/action representation is actually sufficient.

'''

'''
It also exposes a problem with our current proposed state

We previously considered: $$ S_t=[D,C_t] $$
where:
\(D\) = 24 detector bits
\(C_t\) = 9-bit current correction frame.

That gives 33 state dimensions.

The basic idea is still reasonable, but we shouldn't freeze it yet.

The \(512\)-pattern exhaustive experiment will tell us whether the detector representation we're currently using is enough for correction.

Eventually, because the actual project uses noisy multi-round syndrome information, we may want the observation to preserve some temporal structure, e.g.

$$ D_1,D_2,\ldots,D_T $$

rather than blindly treating the 24 bits as an unordered vector.

'''

In [20]:
'''
This will enumerate every possible combination of \(Z\) errors on the 9 data qubits:

$$ 2^9=512. $$

For each error pattern we will record:

physical error pattern;
number of physical \(Z\) errors;
24-bit detector syndrome;
syndrome weight;
logical observable \(L_0\).
'''


# ============================================================
# EXHAUSTIVE Z-ERROR ANALYSIS
#
# Enumerate all 2^9 = 512 combinations of Z errors on the
# nine data qubits.
#
# For every physical error pattern:
#   physical Z pattern
#          ↓
#   Stim circuit
#          ↓
#   raw measurements
#          ↓
#   CLEAN reference converter
#          ↓
#   detector syndrome + logical observable
# ============================================================

print("=" * 70)
print("EXHAUSTIVE Z-ERROR ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Clean reference circuit
# ------------------------------------------------------------

clean_circuit = build_surface_code_circuit(
    distance=DISTANCE,
    rounds=ROUNDS,
    noise_rate=0.0,
)

converter = clean_circuit.compile_m2d_converter()

print(f"Distance              : d = {DISTANCE}")
print(f"Rounds                : {ROUNDS}")
print(f"Data qubits           : {NUM_DATA_QUBITS}")
print(f"Possible Z patterns   : {2 ** NUM_DATA_QUBITS}")
print(f"Detectors             : {clean_circuit.num_detectors}")
print(f"Observables           : {clean_circuit.num_observables}")


# ------------------------------------------------------------
# 2. Insert arbitrary Z-error pattern
# ------------------------------------------------------------

def insert_z_errors_after_first_tick(
    circuit: stim.Circuit,
    error_pattern: np.ndarray,
    data_qubits: list,
) -> stim.Circuit:
    """
    Insert Z errors after the first TICK according to
    a binary error pattern.

    error_pattern[i] = 1 means apply Z to data_qubits[i].
    """

    error_pattern = np.asarray(error_pattern, dtype=np.uint8)

    if len(error_pattern) != len(data_qubits):
        raise ValueError(
            "Error pattern length must match number of data qubits."
        )

    circuit_text = str(circuit)

    marker = "TICK\n"
    position = circuit_text.find(marker)

    if position == -1:
        raise RuntimeError("Could not find the first TICK.")

    insert_position = position + len(marker)

    z_operations = []

    for bit, qubit in zip(error_pattern, data_qubits):
        if bit:
            z_operations.append(f"Z {qubit}\n")

    insertion = "".join(z_operations)

    modified_text = (
        circuit_text[:insert_position]
        + insertion
        + circuit_text[insert_position:]
    )

    return stim.Circuit(modified_text)


# ------------------------------------------------------------
# 3. Convert raw measurements using CLEAN reference
# ------------------------------------------------------------

def get_detector_result(
    fault_circuit: stim.Circuit,
):
    """
    Sample one raw measurement record from fault_circuit,
    then convert it using the CLEAN reference converter.
    """

    sampler = fault_circuit.compile_sampler()

    raw = sampler.sample(shots=1)

    detectors, observables = converter.convert(
        measurements=raw,
        separate_observables=True,
    )

    return (
        raw[0].astype(np.uint8),
        detectors[0].astype(np.uint8),
        observables[0].astype(np.uint8),
    )


# ------------------------------------------------------------
# 4. Enumerate all 512 error patterns
# ------------------------------------------------------------

all_results = []

for error_int in range(2 ** NUM_DATA_QUBITS):

    # Convert integer into a 9-bit binary vector.
    error_pattern = np.array(
        [
            (error_int >> i) & 1
            for i in range(NUM_DATA_QUBITS)
        ],
        dtype=np.uint8,
    )

    fault_circuit = insert_z_errors_after_first_tick(
        clean_circuit,
        error_pattern,
        DATA_QUBITS,
    )

    raw, detectors, observables = get_detector_result(
        fault_circuit
    )

    syndrome_weight = int(detectors.sum())
    physical_weight = int(error_pattern.sum())
    logical_observable = int(observables[0])

    all_results.append(
        {
            "error_int": error_int,
            "error_pattern": error_pattern.copy(),
            "physical_weight": physical_weight,
            "detectors": detectors.copy(),
            "syndrome_weight": syndrome_weight,
            "logical_observable": logical_observable,
        }
    )


print("\nEnumeration complete.")
print(f"Patterns analyzed: {len(all_results)}")


# ------------------------------------------------------------
# 5. Verify the zero-error case
# ------------------------------------------------------------

zero_error_result = all_results[0]

print("\n" + "-" * 70)
print("ZERO-ERROR CHECK")
print("-" * 70)

print(
    "Physical error :",
    zero_error_result["error_pattern"]
)

print(
    "Detector weight:",
    zero_error_result["syndrome_weight"]
)

print(
    "Logical L0     :",
    zero_error_result["logical_observable"]
)

if (
    zero_error_result["syndrome_weight"] == 0
    and zero_error_result["logical_observable"] == 0
):
    print("PASS: zero-error case behaves correctly.")
else:
    print("WARNING: zero-error case is unexpected.")


# ------------------------------------------------------------
# 6. Print all single-Z results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SINGLE-Z ERROR RESULTS")
print("=" * 70)

for result in all_results:

    if result["physical_weight"] != 1:
        continue

    error_pattern = result["error_pattern"]

    qubit_index = int(np.flatnonzero(error_pattern)[0])
    qubit = DATA_QUBITS[qubit_index]

    changed_detectors = np.flatnonzero(
        result["detectors"]
    )

    print(
        f"Z q{qubit:<2} "
        f"{str(DATA_COORDS[qubit]):<10} -> "
        f"D{changed_detectors.tolist()} "
        f"| L0={result['logical_observable']}"
    )


# ------------------------------------------------------------
# 7. Count unique syndromes
# ------------------------------------------------------------

syndrome_to_errors = {}

for result in all_results:

    syndrome_tuple = tuple(
        result["detectors"].tolist()
    )

    syndrome_to_errors.setdefault(
        syndrome_tuple,
        []
    ).append(result["error_pattern"])


unique_syndromes = len(syndrome_to_errors)

print("\n" + "=" * 70)
print("SYNDROME DEGENERACY")
print("=" * 70)

print(
    f"Physical error patterns : {len(all_results)}"
)

print(
    f"Unique detector syndromes: {unique_syndromes}"
)

print(
    f"Degeneracy ratio         : "
    f"{len(all_results) / unique_syndromes:.3f}"
)


# ------------------------------------------------------------
# 8. Find syndromes produced by multiple physical errors
# ------------------------------------------------------------

degenerate_groups = [
    (syndrome, errors)
    for syndrome, errors in syndrome_to_errors.items()
    if len(errors) > 1
]

print(
    f"\nDegenerate syndrome groups: "
    f"{len(degenerate_groups)}"
)

# Show the first few groups.
for group_index, (syndrome, errors) in enumerate(
    degenerate_groups[:10]
):

    print("\n" + "-" * 70)
    print(f"DEGENERATE GROUP {group_index + 1}")

    print("Syndrome:")
    print(np.array(syndrome, dtype=np.uint8))

    print("Number of physical errors:", len(errors))

    for error in errors[:10]:

        qubits = [
            DATA_QUBITS[i]
            for i, bit in enumerate(error)
            if bit
        ]

        print(
            f"  {error.tolist()} -> "
            f"Z qubits {qubits}"
        )

    if len(errors) > 10:
        print(
            f"  ... {len(errors) - 10} more"
        )


# ------------------------------------------------------------
# 9. Logical-class structure of degenerate syndromes
# ------------------------------------------------------------

mixed_logical_groups = []

for syndrome, errors in syndrome_to_errors.items():

    logical_classes = set()

    for error in errors:

        # Locate corresponding result.
        for result in all_results:

            if np.array_equal(
                result["error_pattern"],
                error
            ):
                logical_classes.add(
                    result["logical_observable"]
                )
                break

    if len(logical_classes) > 1:

        mixed_logical_groups.append(
            (
                syndrome,
                errors,
                logical_classes,
            )
        )


print("\n" + "=" * 70)
print("LOGICALLY AMBIGUOUS SYNDROMES")
print("=" * 70)

print(
    "Syndrome groups containing both L0=0 and L0=1 errors:",
    len(mixed_logical_groups)
)

for group_index, (
    syndrome,
    errors,
    logical_classes,
) in enumerate(
    mixed_logical_groups[:10]
):

    print("\n" + "-" * 70)
    print(f"AMBIGUOUS GROUP {group_index + 1}")

    print("Syndrome:")
    print(np.array(syndrome, dtype=np.uint8))

    print("Logical classes:", sorted(logical_classes))

    for error in errors:

        matching_result = next(
            result
            for result in all_results
            if np.array_equal(
                result["error_pattern"],
                error
            )
        )

        qubits = [
            DATA_QUBITS[i]
            for i, bit in enumerate(error)
            if bit
        ]

        print(
            f"  Z qubits {qubits} "
            f"-> L0={matching_result['logical_observable']}"
        )


# ------------------------------------------------------------
# 10. Summary by physical-error weight
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY BY PHYSICAL ERROR WEIGHT")
print("=" * 70)

for weight in range(NUM_DATA_QUBITS + 1):

    subset = [
        result
        for result in all_results
        if result["physical_weight"] == weight
    ]

    if not subset:
        continue

    logical_1_count = sum(
        result["logical_observable"] == 1
        for result in subset
    )

    unique_syndromes_for_weight = len(
        {
            tuple(result["detectors"].tolist())
            for result in subset
        }
    )

    print(
        f"weight={weight}: "
        f"errors={len(subset):3d}, "
        f"unique syndromes={unique_syndromes_for_weight:3d}, "
        f"L0=1={logical_1_count:3d}"
    )


print("\n" + "=" * 70)
print("EXHAUSTIVE ANALYSIS COMPLETE")
print("=" * 70)

EXHAUSTIVE Z-ERROR ANALYSIS
Distance              : d = 3
Rounds                : 3
Data qubits           : 9
Possible Z patterns   : 512
Detectors             : 24
Observables           : 1

Enumeration complete.
Patterns analyzed: 512

----------------------------------------------------------------------
ZERO-ERROR CHECK
----------------------------------------------------------------------
Physical error : [0 0 0 0 0 0 0 0 0]
Detector weight: 0
Logical L0     : 0
PASS: zero-error case behaves correctly.

SINGLE-Z ERROR RESULTS
Z q1  (1, 1)     -> D[0] | L0=1
Z q3  (3, 1)     -> D[0, 2] | L0=0
Z q5  (5, 1)     -> D[2] | L0=0
Z q8  (1, 3)     -> D[1] | L0=1
Z q10 (3, 3)     -> D[1, 2] | L0=0
Z q12 (5, 3)     -> D[2] | L0=0
Z q15 (1, 5)     -> D[1] | L0=1
Z q17 (3, 5)     -> D[1, 3] | L0=0
Z q19 (5, 5)     -> D[3] | L0=0

SYNDROME DEGENERACY
Physical error patterns : 512
Unique detector syndromes: 16
Degeneracy ratio         : 32.000

Degenerate syndrome groups: 16

------------------

In [ ]:
'''
There are three particularly important outputs.

A. Unique syndrome count

We have 512 physical error configurations, but a QEC code is deliberately degenerate: multiple physical errors can produce the same syndrome.

So we expect:

$$ N_{\text{unique syndromes}} < 512. $$

That's not a problem—it is fundamental to QEC.

B. Degenerate groups with the same logical class

For example, several errors may give the same syndrome and all have \(L_0=0\).

A decoder can choose any correction that returns the system to the correct logical class.

C. Degenerate groups containing both \(L_0=0\) and \(L_0=1\)

This is especially important.

If the same syndrome can correspond to both logical classes, then the syndrome alone cannot uniquely identify the logical state.

That is where minimum-weight decoding / learned decoding becomes meaningful.

'''

'''
We should not assume that “correction = exact physical error” is the only definition of success.

In QEC, a correction does not necessarily need to reproduce the exact inverse physical error. It only needs to make the combined residual error logically equivalent to the identity for the encoded state.

That distinction will be essential when we implement the RL reward.
'''

In [ ]:
'''
                 hidden physical error
                         │
                         ▼
                  detector syndrome
                         │
                         ▼
                  RL decoder
                         │
                         ▼
                    correction
                         │
                         ▼
              residual logical error?

    The agent does not need to reconstruct the exact physical error.

For example, suppose:

hidden error = Z1 Z3 Z5
syndrome    = 000...0
L0          = 1

The agent could potentially choose some other correction \(C\) such that

$$ E C $$

is stabilizer-equivalent to the identity.

So our eventual reward must evaluate logical correctness, not exact physical-error reconstruction.

'''

'''
Our current experiment injected all errors:

after the first TICK

Therefore, although the circuit has

$$ 24 $$

detectors, this particular experiment only gives information in the first four detector positions:

D0 D1 D2 D3

The other 20 detectors remain zero.

This means we should not yet build the RL environment around these 16 syndromes.

The experiment has successfully validated the mechanics, but the final RL environment needs errors occurring throughout the noisy QEC process.
'''

'''
Before introducing random noise and temporal errors, let's build the logical correction evaluator.

We can exploit the exhaustive table we just generated.

For every physical error \(E\) and candidate correction \(C\):

$$ E_{\mathrm{res}}=E\oplus C. $$

Then we look up the logical class of \(E_{\mathrm{res}}\).

For our restricted \(Z\)-only experiment:
$$ L(E_{\mathrm{res}})=0 $$ means successful correction.
'''

In [21]:
# ============================================================
# CORRECTION / REWARD VALIDATION
#
# For the current controlled Z-only experiment:
#
#   hidden physical error E
#            +
#   proposed correction C
#            ↓
#   residual error E XOR C
#            ↓
#   logical class of residual
#
# Success  : residual logical class = 0
# Failure  : residual logical class = 1
#
# IMPORTANT:
# This is a validation environment only.
# It is NOT the final noisy RL environment.
# ============================================================

print("=" * 70)
print("CORRECTION / REWARD VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Build lookup table:
#
# physical error pattern -> logical observable
# ------------------------------------------------------------

logical_lookup = {}

for result in all_results:

    error_tuple = tuple(
        result["error_pattern"].tolist()
    )

    logical_lookup[error_tuple] = (
        result["logical_observable"]
    )


print(
    f"Logical lookup entries: {len(logical_lookup)}"
)


# ------------------------------------------------------------
# 2. Helper: physical error -> syndrome
# ------------------------------------------------------------

syndrome_lookup = {}

for result in all_results:

    error_tuple = tuple(
        result["error_pattern"].tolist()
    )

    syndrome_tuple = tuple(
        result["detectors"].tolist()
    )

    syndrome_lookup[error_tuple] = syndrome_tuple


# ------------------------------------------------------------
# 3. Correction evaluator
# ------------------------------------------------------------

def evaluate_z_correction(
    physical_error: np.ndarray,
    correction: np.ndarray,
):
    """
    Evaluate a proposed Z correction.

    Parameters
    ----------
    physical_error:
        9-bit vector describing hidden physical Z errors.

    correction:
        9-bit vector describing applied Z corrections.

    Returns
    -------
    result : dict
        Contains residual error, logical class and success.
    """

    physical_error = np.asarray(
        physical_error,
        dtype=np.uint8,
    )

    correction = np.asarray(
        correction,
        dtype=np.uint8,
    )

    if len(physical_error) != NUM_DATA_QUBITS:
        raise ValueError(
            "physical_error must have length "
            f"{NUM_DATA_QUBITS}."
        )

    if len(correction) != NUM_DATA_QUBITS:
        raise ValueError(
            "correction must have length "
            f"{NUM_DATA_QUBITS}."
        )

    # Z * Z = I
    residual = np.bitwise_xor(
        physical_error,
        correction,
    )

    residual_tuple = tuple(
        residual.tolist()
    )

    logical_class = logical_lookup[
        residual_tuple
    ]

    success = (
        logical_class == 0
    )

    return {
        "physical_error": physical_error.copy(),
        "correction": correction.copy(),
        "residual_error": residual.copy(),
        "logical_class": int(logical_class),
        "success": bool(success),
    }


# ------------------------------------------------------------
# 4. Test 1: no error + no correction
# ------------------------------------------------------------

zero = np.zeros(
    NUM_DATA_QUBITS,
    dtype=np.uint8,
)

result = evaluate_z_correction(
    physical_error=zero,
    correction=zero,
)

print("\n" + "-" * 70)
print("TEST 1 — NO ERROR / NO CORRECTION")
print("-" * 70)

print("Physical error :", result["physical_error"])
print("Correction     :", result["correction"])
print("Residual       :", result["residual_error"])
print("Logical class  :", result["logical_class"])
print("Success        :", result["success"])


# ------------------------------------------------------------
# 5. Test 2: q3 error + q3 correction
# ------------------------------------------------------------

physical_q3 = np.zeros(
    NUM_DATA_QUBITS,
    dtype=np.uint8,
)

physical_q3[
    DATA_QUBITS.index(3)
] = 1

correction_q3 = physical_q3.copy()

result = evaluate_z_correction(
    physical_error=physical_q3,
    correction=correction_q3,
)

print("\n" + "-" * 70)
print("TEST 2 — Z(q3) / CORRECT Z(q3)")
print("-" * 70)

print("Physical error :", result["physical_error"])
print("Correction     :", result["correction"])
print("Residual       :", result["residual_error"])
print("Logical class  :", result["logical_class"])
print("Success        :", result["success"])


# ------------------------------------------------------------
# 6. Test 3: q1 error + q1 correction
# ------------------------------------------------------------

physical_q1 = np.zeros(
    NUM_DATA_QUBITS,
    dtype=np.uint8,
)

physical_q1[
    DATA_QUBITS.index(1)
] = 1

correction_q1 = physical_q1.copy()

result = evaluate_z_correction(
    physical_error=physical_q1,
    correction=correction_q1,
)

print("\n" + "-" * 70)
print("TEST 3 — Z(q1) / CORRECT Z(q1)")
print("-" * 70)

print("Physical error :", result["physical_error"])
print("Correction     :", result["correction"])
print("Residual       :", result["residual_error"])
print("Logical class  :", result["logical_class"])
print("Success        :", result["success"])


# ------------------------------------------------------------
# 7. Test 4: q1 error + WRONG q3 correction
# ------------------------------------------------------------

result = evaluate_z_correction(
    physical_error=physical_q1,
    correction=physical_q3,
)

print("\n" + "-" * 70)
print("TEST 4 — Z(q1) / WRONG Z(q3)")
print("-" * 70)

print("Physical error :", result["physical_error"])
print("Correction     :", result["correction"])
print("Residual       :", result["residual_error"])
print("Logical class  :", result["logical_class"])
print("Success        :", result["success"])


# ------------------------------------------------------------
# 8. Test 5: logical-equivalent correction
#
# q1 is itself logical according to the current experiment.
# q1 + q8 has L0 = 0.
# ------------------------------------------------------------

correction_q1_q8 = np.zeros(
    NUM_DATA_QUBITS,
    dtype=np.uint8,
)

correction_q1_q8[
    DATA_QUBITS.index(1)
] = 1

correction_q1_q8[
    DATA_QUBITS.index(8)
] = 1

result = evaluate_z_correction(
    physical_error=physical_q1,
    correction=correction_q1_q8,
)

print("\n" + "-" * 70)
print("TEST 5 — Z(q1) / CORRECTION Z(q1)Z(q8)")
print("-" * 70)

print("Physical error :", result["physical_error"])
print("Correction     :", result["correction"])
print("Residual       :", result["residual_error"])
print("Logical class  :", result["logical_class"])
print("Success        :", result["success"])


# ------------------------------------------------------------
# 9. Test correction involution
#
# Applying the same correction twice cancels:
#
#       C XOR C = 0
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

involution_passed = True

for _ in range(100):

    correction = rng.integers(
        0,
        2,
        size=NUM_DATA_QUBITS,
        dtype=np.uint8,
    )

    twice = np.bitwise_xor(
        correction,
        correction,
    )

    if not np.array_equal(
        twice,
        zero,
    ):
        involution_passed = False
        break


print("\n" + "-" * 70)
print("TEST 6 — Z² = I / XOR CANCELLATION")
print("-" * 70)

if involution_passed:
    print(
        "PASS: applying the same Z correction twice "
        "cancels exactly."
    )
else:
    print(
        "FAIL: correction cancellation test failed."
    )


# ------------------------------------------------------------
# 10. Exhaustive correction-success statistics
#
# For every physical error E, test every possible correction C.
#
# This tells us how many corrections are logically valid.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXHAUSTIVE CORRECTION VALIDITY")
print("=" * 70)

total_pairs = 0
successful_pairs = 0

success_count_by_error_weight = {
    weight: []
    for weight in range(NUM_DATA_QUBITS + 1)
}

for physical_result in all_results:

    physical_error = physical_result[
        "error_pattern"
    ]

    physical_weight = int(
        physical_error.sum()
    )

    successful_for_this_error = 0

    for correction_result in all_results:

        correction = correction_result[
            "error_pattern"
        ]

        result = evaluate_z_correction(
            physical_error=physical_error,
            correction=correction,
        )

        total_pairs += 1

        if result["success"]:
            successful_pairs += 1
            successful_for_this_error += 1

    success_count_by_error_weight[
        physical_weight
    ].append(
        successful_for_this_error
    )


print(
    f"Physical errors tested : {len(all_results)}"
)

print(
    f"Corrections per error   : {len(all_results)}"
)

print(
    f"Total (E,C) pairs       : {total_pairs}"
)

print(
    f"Successful pairs        : {successful_pairs}"
)

print(
    f"Successful fraction     : "
    f"{successful_pairs / total_pairs:.4f}"
)


# ------------------------------------------------------------
# 11. Success statistics by physical error weight
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SUCCESSFUL CORRECTIONS BY PHYSICAL ERROR WEIGHT")
print("-" * 70)

for weight in range(NUM_DATA_QUBITS + 1):

    values = success_count_by_error_weight[
        weight
    ]

    if not values:
        continue

    print(
        f"weight={weight}: "
        f"min={min(values):3d}, "
        f"max={max(values):3d}, "
        f"mean={np.mean(values):6.2f}"
    )


print("\n" + "=" * 70)
print("CORRECTION VALIDATION COMPLETE")
print("=" * 70)

CORRECTION / REWARD VALIDATION
Logical lookup entries: 512

----------------------------------------------------------------------
TEST 1 — NO ERROR / NO CORRECTION
----------------------------------------------------------------------
Physical error : [0 0 0 0 0 0 0 0 0]
Correction     : [0 0 0 0 0 0 0 0 0]
Residual       : [0 0 0 0 0 0 0 0 0]
Logical class  : 0
Success        : True

----------------------------------------------------------------------
TEST 2 — Z(q3) / CORRECT Z(q3)
----------------------------------------------------------------------
Physical error : [0 1 0 0 0 0 0 0 0]
Correction     : [0 1 0 0 0 0 0 0 0]
Residual       : [0 0 0 0 0 0 0 0 0]
Logical class  : 0
Success        : True

----------------------------------------------------------------------
TEST 3 — Z(q1) / CORRECT Z(q1)
----------------------------------------------------------------------
Physical error : [1 0 0 0 0 0 0 0 0]
Correction     : [1 0 0 0 0 0 0 0 0]
Residual       : [0 0 0 0 0 0 0 0 0]
L

In [ ]:
'''
Our eventual RL reward should not simply be:

+1 if correction == physical_error
-1 otherwise

Instead, it should be something closer to:

successful logical correction → positive terminal reward
failed logical correction      → negative terminal reward
unnecessary correction         → small penalty
STOP on successful state       → positive reward
STOP on failed state           → negative reward

That gives the agent freedom to discover a correction that is logically equivalent, rather than forcing it to memorize the exact hidden physical error.
'''

'''
Residual-error formulation is correct

For Z-only errors, using

$$ R = E \oplus C $$

is correct because \(Z^2=I\). Your tests confirm this:

\(E=Z_{q3}, C=Z_{q3}\Rightarrow R=I\)
\(E=Z_{q1}, C=Z_{q1}\Rightarrow R=I\)
applying the same correction twice cancels.
Logical success is correctly defined

The important distinction is:

$$ \boxed{\text{Success} \iff L(R)=0} $$

rather than

$$ C=E. $$

Your Test 5 demonstrates why this matters. Z(q1) followed by Z(q1)Z(q8) leaves Z(q8), which is logically nontrivial in this particular code experiment, so it correctly fails.

The exhaustive result is particularly useful

You tested

$$ 512\times512=262\,144 $$

error/correction combinations and obtained exactly

$$ 131\,072 = \frac{262\,144}{2}. $$

So:

$$ \boxed{P(\text{logical success over all corrections})=0.5} $$

and, remarkably, every physical error weight has exactly 256 successful corrections.

That is not a bug. It follows from the current lookup structure: the 512 possible corrections split evenly into the two logical classes, and XOR with a fixed physical error simply permutes those 512 corrections.

But this is also the point where we should stop using the exhaustive table as the RL environment

This experiment validates the mathematics of the reward evaluator, not the usefulness of the environment for learning.

The current setup has an important limitation:

All physical Z errors were inserted at one controlled location in the circuit.

Consequently, only the first four detector positions carry information about those injected faults. We already saw that this produced only 16 unique detector syndromes out of 512 physical errors.

So if we trained an RL agent directly on this setup, it would be learning an artificially small problem rather than learning genuine spacetime syndrome decoding.

'''

In [ ]:
'''
                    ┌──────────────────────┐
                │ Physical error/fault │
                │      (hidden)        │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ Noisy surface-code   │
                │      circuit         │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ 24-bit detector      │
                │ spacetime syndrome   │
                └──────────┬───────────┘
                           │
                           ▼
                     RL Agent
                           │
                    action: Z(qi)
                           │
                           ▼
                ┌──────────────────────┐
                │ accumulated          │
                │ correction frame C   │
                └──────────┬───────────┘
                           │
                      STOP action
                           │
                           ▼
                ┌──────────────────────┐
                │ hidden E ⊕ C         │
                │ logical evaluation   │
                └──────────┬───────────┘
                           │
                           ▼
                    Reward / Success

    '''

'''
The key research property is:

Agent observes
$$ s = \text{detector syndrome} $$

but does not observe

$$ E,\quad L(E),\quad\text{or the true physical error}. $$

The hidden error is used only by the environment to calculate the reward.

Observation design

For \(d=3\), we currently have 24 detectors:

$$ 24 = 3\times8. $$

Rather than immediately treating them as an arbitrary 24-bit vector, I suggest preserving their spacetime structure:

$$ S = \begin{bmatrix} s_{0,0}&\cdots&s_{0,7}\\ s_{1,0}&\cdots&s_{1,7}\\ s_{2,0}&\cdots&s_{2,7} \end{bmatrix}. $$

The neural network can initially flatten this to 24 inputs:

observation = [D0, D1, ..., D23]

but we retain the round structure internally so that later we can experiment with architectures that explicitly exploit spacetime locality.
'''

'''
Action space

Your current action space is actually a good first RL action space:

0 → Z(q1)
1 → Z(q3)
2 → Z(q5)
3 → Z(q8)
4 → Z(q10)
5 → Z(q12)
6 → Z(q15)
7 → Z(q17)
8 → Z(q19)
9 → STOP

An action doesn't immediately terminate the episode.

Instead: $$ C_{t+1}=C_t\oplus Z_i. $$

For example:
    
    Initial C:
    000000000
    
    Agent: Z(q1)
    001000000  # conceptually
    
    Agent: Z(q8)
    100100000
    
    Agent: Z(q1)
    000100000

because repeated corrections cancel.

STOP ends the episode and triggers the final logical-success evaluation.

Reward

For the first version, I would keep the reward deliberately simple.

For a correction action: $$ r_{\text{step}}=-\lambda $$

with a small correction cost, e.g. $$ \lambda=0.01. $$

When the agent chooses STOP:

$$ r_{\text{terminal}}= \begin{cases} +1 & L(E\oplus C)=0\\ -1 & L(E\oplus C)=1. \end{cases} $$

Thus:
    Z(q1)
    Z(q8)
    STOP

might produce:
    -0.01
    -0.01
    +1.00

while an unsuccessful correction gives:
    -0.01
    -0.01
    -1.00

This gives the agent two competing objectives:

decode successfully
avoid unnecessary corrections

which is much more meaningful than rewarding exact reconstruction of the hidden physical error.
'''

In [ ]:
'''

The first RL environment.
We will first build a controlled RL environment from the validated 512-error dataset.
This is a development environment, not the final noisy QEC environment.
We will be validating the environment before training and then moving toward the full stochastic/spacetime problem.

The first environment will let us verify that:
    observations are correct;
    hidden physical errors stay hidden;
    correction actions compose correctly;
    STOP works;
    rewards are correct;
    episodes terminate correctly;
    a random policy can run;
    eventually DQN can interact with the environment.

'''


In [ ]:
'''
6.1 Environment formulation

For the first version:

$$ \boxed{s_t=[D,C_t]} $$

where:

\(D\) = observed 24-bit detector syndrome;
\(C_t\) = 9-bit accumulated Z-correction frame.

The hidden physical error is:

$$ E\in\{0,1\}^9 $$

but the agent never receives \(E\).

The environment internally evaluates:

$$ R_{\mathrm{residual}}=E\oplus C_t. $$

The action space remains:

$$ A=\{Z_{q_1},\ldots,Z_{q_9},STOP\}. $$

This follows the correction-state formulation: the syndrome remains fixed while the agent constructs a correction configuration.
'''

In [26]:
# ============================================================
# BUILD LOGICAL LOOKUP DIRECTLY FROM all_results
# ============================================================

logical_lookup = {}

for result in all_results:

    error = np.asarray(
        result["error_pattern"],
        dtype=np.uint8
    )

    logical_class = int(
        result["logical_observable"]
    )

    key = tuple(error.tolist())

    logical_lookup[key] = logical_class


print("=" * 70)
print("LOGICAL LOOKUP")
print("=" * 70)

print(f"Lookup entries : {len(logical_lookup)}")

assert len(logical_lookup) == 512

print("PASS: all 512 physical error patterns are represented.")

LOGICAL LOOKUP
Lookup entries : 512
PASS: all 512 physical error patterns are represented.


In [27]:
# ============================================================
# STAGE 6 — RL ENVIRONMENT CONFIGURATION
# ============================================================

import numpy as np

print("=" * 70)
print("RL ENVIRONMENT CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Dimensions
# ------------------------------------------------------------

# Number of detector bits
NUM_DETECTORS = len(all_results[0]["detectors"])

# Number of controllable data qubits
NUM_DATA_QUBITS = len(DATA_QUBITS)

# Observation:
#
#   [24 detector bits | 9 correction-frame bits]
#
# Total = 33
OBSERVATION_DIM = NUM_DETECTORS + NUM_DATA_QUBITS

# Actions:
#
#   0 ... 8 -> Z correction on one data qubit
#   9       -> STOP
NUM_ACTIONS = NUM_DATA_QUBITS + 1
STOP_ACTION = NUM_DATA_QUBITS

# Maximum number of correction actions before
# the environment forces termination.
MAX_STEPS = NUM_DATA_QUBITS

print(f"Detector bits       : {NUM_DETECTORS}")
print(f"Data qubits         : {NUM_DATA_QUBITS}")
print(f"Observation dim     : {OBSERVATION_DIM}")
print(f"Number of actions   : {NUM_ACTIONS}")
print(f"STOP action index   : {STOP_ACTION}")
print(f"Maximum steps       : {MAX_STEPS}")

RL ENVIRONMENT CONFIGURATION
Detector bits       : 24
Data qubits         : 9
Observation dim     : 33
Number of actions   : 10
STOP action index   : 9
Maximum steps       : 9


In [28]:
# ============================================================
# BUILD CONTROLLED RL DATASET
# ============================================================

ENV_ERRORS = []
ENV_SYNDROMES = []
ENV_LOGICAL_CLASSES = []

for result in all_results:

    # Hidden physical Z-error pattern
    error = np.asarray(
        result["error_pattern"],
        dtype=np.uint8
    )

    # Detector syndrome observed by the agent
    syndrome = np.asarray(
        result["detectors"],
        dtype=np.uint8
    )

    # Hidden logical observable used only for evaluation
    logical_class = int(
        result["logical_observable"]
    )

    ENV_ERRORS.append(error)
    ENV_SYNDROMES.append(syndrome)
    ENV_LOGICAL_CLASSES.append(logical_class)


ENV_ERRORS = np.asarray(
    ENV_ERRORS,
    dtype=np.uint8
)

ENV_SYNDROMES = np.asarray(
    ENV_SYNDROMES,
    dtype=np.uint8
)

ENV_LOGICAL_CLASSES = np.asarray(
    ENV_LOGICAL_CLASSES,
    dtype=np.uint8
)


print("=" * 70)
print("CONTROLLED RL DATASET")
print("=" * 70)

print(f"Physical errors    : {len(ENV_ERRORS)}")
print(f"Error shape        : {ENV_ERRORS.shape}")
print(f"Syndrome shape     : {ENV_SYNDROMES.shape}")
print(f"Logical labels     : {ENV_LOGICAL_CLASSES.shape}")

print()
print("Logical class distribution:")

unique, counts = np.unique(
    ENV_LOGICAL_CLASSES,
    return_counts=True
)

for label, count in zip(unique, counts):
    print(f"  L0={label}: {count}")

CONTROLLED RL DATASET
Physical errors    : 512
Error shape        : (512, 9)
Syndrome shape     : (512, 24)
Logical labels     : (512,)

Logical class distribution:
  L0=0: 256
  L0=1: 256


In [29]:
# ============================================================
# SURFACE CODE CORRECTION ENVIRONMENT
# ============================================================

class SurfaceCodeCorrectionEnv:
    """
    Controlled RL environment for Z-only correction decoding.

    The environment samples a hidden physical Z-error E and
    exposes only its detector syndrome D to the agent.

    The agent sequentially constructs a correction C.

    State / observation:
        [D, C]

    where:
        D = fixed detector syndrome
        C = current correction frame

    Action:
        0 ... 8 -> apply Z correction to corresponding data qubit
        9       -> STOP

    At STOP:
        residual = E XOR C

        logical_success = logical_class(residual) == 0

        reward:
            +1 for logical success
            -1 for logical failure

    This is a controlled development environment.
    It uses the previously validated exhaustive 512-error table.
    """

    def __init__(
        self,
        errors,
        syndromes,
        logical_lookup,
        data_qubits,
        max_steps=9,
        seed=42,
    ):
        self.errors = np.asarray(errors, dtype=np.uint8)
        self.syndromes = np.asarray(syndromes, dtype=np.uint8)
        self.logical_lookup = logical_lookup
        self.data_qubits = list(data_qubits)

        self.num_errors = len(self.errors)
        self.num_data_qubits = len(self.data_qubits)
        self.num_detectors = self.syndromes.shape[1]

        self.num_actions = self.num_data_qubits + 1
        self.stop_action = self.num_data_qubits

        self.max_steps = max_steps

        self.rng = np.random.default_rng(seed)

        # Episode state
        self.error = None
        self.syndrome = None
        self.correction = None
        self.step_count = 0
        self.done = False

    # --------------------------------------------------------
    # Helper functions
    # --------------------------------------------------------

    def _make_observation(self):
        """
        Construct the observation visible to the agent.

        IMPORTANT:
        self.error is deliberately NOT included.
        """
        observation = np.concatenate([
            self.syndrome,
            self.correction,
        ])

        return observation.astype(np.float32)

    def _logical_class(self, error):
        """
        Return the logical class of a 9-bit Z error pattern.
        """
        key = tuple(np.asarray(error, dtype=np.uint8).tolist())
        return int(self.logical_lookup[key])

    # --------------------------------------------------------
    # Reset
    # --------------------------------------------------------

    def reset(self, error_index=None):
        """
        Start a new episode.

        If error_index is supplied, use that exact hidden error.
        Otherwise sample uniformly from the 512 physical errors.
        """

        if error_index is None:
            error_index = self.rng.integers(
                0,
                self.num_errors
            )

        error_index = int(error_index)

        self.error = self.errors[error_index].copy()
        self.syndrome = self.syndromes[error_index].copy()

        # No correction has been applied yet.
        self.correction = np.zeros(
            self.num_data_qubits,
            dtype=np.uint8
        )

        self.step_count = 0
        self.done = False

        observation = self._make_observation()

        info = {
            "error_index": error_index,
            "step": self.step_count,
        }

        return observation, info

    # --------------------------------------------------------
    # Step
    # --------------------------------------------------------

    def step(self, action):
        """
        Apply one RL action.

        Returns:
            observation
            reward
            terminated
            truncated
            info
        """

        if self.done:
            raise RuntimeError(
                "Episode has already terminated. "
                "Call reset() before step()."
            )

        action = int(action)

        if action < 0 or action >= self.num_actions:
            raise ValueError(
                f"Invalid action {action}. "
                f"Valid actions are 0..{self.num_actions - 1}."
            )

        self.step_count += 1

        # ----------------------------------------------------
        # STOP
        # ----------------------------------------------------

        if action == self.stop_action:

            residual = np.bitwise_xor(
                self.error,
                self.correction
            )

            logical_class = self._logical_class(residual)

            logical_success = (
                logical_class == 0
            )

            reward = 1.0 if logical_success else -1.0

            self.done = True

            observation = self._make_observation()

            info = {
                "action_name": "STOP",
                "correction": self.correction.copy(),
                "residual": residual.copy(),
                "logical_class": logical_class,
                "logical_success": logical_success,
                "step": self.step_count,
            }

            return (
                observation,
                reward,
                True,
                False,
                info,
            )

        # ----------------------------------------------------
        # Correction action
        # ----------------------------------------------------

        self.correction[action] ^= 1

        # Non-terminal correction receives zero reward
        reward = 0.0

        # Forced termination if correction budget is exhausted
        truncated = self.step_count >= self.max_steps

        if truncated:

            residual = np.bitwise_xor(
                self.error,
                self.correction
            )

            logical_class = self._logical_class(residual)

            logical_success = (
                logical_class == 0
            )

            reward = 1.0 if logical_success else -1.0

            self.done = True

            observation = self._make_observation()

            info = {
                "action_name": ACTION_NAMES[action],
                "correction": self.correction.copy(),
                "residual": residual.copy(),
                "logical_class": logical_class,
                "logical_success": logical_success,
                "step": self.step_count,
                "forced_termination": True,
            }

            return (
                observation,
                reward,
                False,
                True,
                info,
            )

        observation = self._make_observation()

        info = {
            "action_name": ACTION_NAMES[action],
            "correction": self.correction.copy(),
            "step": self.step_count,
        }

        return (
            observation,
            reward,
            False,
            False,
            info,
        )

In [30]:
# ============================================================
# CREATE ENVIRONMENT
# ============================================================

env = SurfaceCodeCorrectionEnv(
    errors=ENV_ERRORS,
    syndromes=ENV_SYNDROMES,
    logical_lookup=logical_lookup,
    data_qubits=DATA_QUBITS,
    max_steps=MAX_STEPS,
    seed=SEED,
)

print("=" * 70)
print("ENVIRONMENT CREATED")
print("=" * 70)

print(f"Observation dimension : {OBSERVATION_DIM}")
print(f"Action count          : {env.num_actions}")
print(f"STOP action            : {env.stop_action}")
print(f"Maximum steps          : {env.max_steps}")

ENVIRONMENT CREATED
Observation dimension : 33
Action count          : 10
STOP action            : 9
Maximum steps          : 9


In [31]:
# ============================================================
# RESET TEST
# ============================================================

state, info = env.reset()

print("=" * 70)
print("RESET TEST")
print("=" * 70)

print(f"State shape : {state.shape}")
print(f"State dtype : {state.dtype}")
print(f"Step        : {info['step']}")

print()
print("Visible state:")
print(state)

print()
print("Hidden error:")
print(env.error)

print()
print("Correction frame:")
print(env.correction)

RESET TEST
State shape : (33,)
State dtype : float32
Step        : 0

Visible state:
[1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Hidden error:
[1 0 1 1 0 1 0 0 0]

Correction frame:
[0 0 0 0 0 0 0 0 0]


In [32]:
# ============================================================
# HIDDEN-ERROR LEAKAGE TEST
# ============================================================

state, info = env.reset(error_index=0)

assert state.shape == (OBSERVATION_DIM,)

# The observation consists of:
#   first 24 -> syndrome
#   last 9   -> correction frame

visible_syndrome = state[:NUM_DETECTORS]
visible_correction = state[NUM_DETECTORS:]

assert np.array_equal(
    visible_syndrome.astype(np.uint8),
    env.syndrome
)

assert np.array_equal(
    visible_correction.astype(np.uint8),
    env.correction
)

# The hidden error is stored internally only.
assert hasattr(env, "error")

print("=" * 70)
print("HIDDEN-ERROR LEAKAGE TEST")
print("=" * 70)

print("PASS")
print()
print("Agent-visible:")
print("  syndrome       :", visible_syndrome.astype(np.uint8))
print("  correction     :", visible_correction.astype(np.uint8))

print()
print("Environment-only:")
print("  physical error :", env.error)

HIDDEN-ERROR LEAKAGE TEST
PASS

Agent-visible:
  syndrome       : [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
  correction     : [0 0 0 0 0 0 0 0 0]

Environment-only:
  physical error : [0 0 0 0 0 0 0 0 0]


In [33]:
# ============================================================
# CORRECTION TRANSITION TEST
# ============================================================

state, info = env.reset(error_index=0)

print("=" * 70)
print("CORRECTION TRANSITION TEST")
print("=" * 70)

print("Initial correction:")
print(env.correction)

# Apply Z(q1)
state, reward, terminated, truncated, info = env.step(0)

print()
print("After Z(q1):")
print(env.correction)

assert np.array_equal(
    env.correction,
    np.array([1, 0, 0, 0, 0, 0, 0, 0, 0], dtype=np.uint8)
)

# Apply Z(q1) again
state, reward, terminated, truncated, info = env.step(0)

print()
print("After Z(q1) again:")
print(env.correction)

assert np.array_equal(
    env.correction,
    np.zeros(NUM_DATA_QUBITS, dtype=np.uint8)
)

print()
print("PASS: repeated correction cancels via XOR.")

CORRECTION TRANSITION TEST
Initial correction:
[0 0 0 0 0 0 0 0 0]

After Z(q1):
[1 0 0 0 0 0 0 0 0]

After Z(q1) again:
[0 0 0 0 0 0 0 0 0]

PASS: repeated correction cancels via XOR.


In [34]:
# ============================================================
# STOP / SUCCESS TEST
# ============================================================

target_error = np.array(
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    dtype=np.uint8
)

matches = np.where(
    np.all(ENV_ERRORS == target_error, axis=1)
)[0]

assert len(matches) == 1

error_index = int(matches[0])

state, info = env.reset(
    error_index=error_index
)

print("=" * 70)
print("STOP / SUCCESS TEST")
print("=" * 70)

print("Hidden error:")
print(env.error)

# Apply Z(q3)
state, reward, terminated, truncated, info = env.step(1)

print()
print("Correction:")
print(env.correction)

print()
print("Reward before STOP:", reward)
print("Terminated:", terminated)

# STOP
state, reward, terminated, truncated, info = env.step(
    STOP_ACTION
)

print()
print("STOP reward       :", reward)
print("Logical class     :", info["logical_class"])
print("Logical success   :", info["logical_success"])
print("Terminated        :", terminated)
print("Residual          :", info["residual"])

assert reward == 1.0
assert info["logical_success"] is True
assert terminated is True

print()
print("PASS")

STOP / SUCCESS TEST
Hidden error:
[0 1 0 0 0 0 0 0 0]

Correction:
[0 1 0 0 0 0 0 0 0]

Reward before STOP: 0.0
Terminated: False

STOP reward       : 1.0
Logical class     : 0
Logical success   : True
Terminated        : True
Residual          : [0 0 0 0 0 0 0 0 0]

PASS


In [35]:
# ============================================================
# STOP / FAILURE TEST
# ============================================================

target_error = np.array(
    [1, 0, 0, 0, 0, 0, 0, 0, 0],
    dtype=np.uint8
)

matches = np.where(
    np.all(ENV_ERRORS == target_error, axis=1)
)[0]

assert len(matches) == 1

error_index = int(matches[0])

state, info = env.reset(
    error_index=error_index
)

# Wrong correction: Z(q3)
state, reward, terminated, truncated, info = env.step(1)

assert reward == 0.0
assert terminated is False

# STOP
state, reward, terminated, truncated, info = env.step(
    STOP_ACTION
)

print("=" * 70)
print("STOP / FAILURE TEST")
print("=" * 70)

print("Hidden error   :", env.error)
print("Correction     :", info["correction"])
print("Residual       :", info["residual"])
print("Logical class  :", info["logical_class"])
print("Success        :", info["logical_success"])
print("Reward         :", reward)

assert reward == -1.0
assert info["logical_success"] is False
assert terminated is True

print()
print("PASS")

STOP / FAILURE TEST
Hidden error   : [1 0 0 0 0 0 0 0 0]
Correction     : [0 1 0 0 0 0 0 0 0]
Residual       : [1 1 0 0 0 0 0 0 0]
Logical class  : 1
Success        : False
Reward         : -1.0

PASS


In [36]:
# ============================================================
# IMMEDIATE STOP TEST
# ============================================================

zero_error = np.zeros(
    NUM_DATA_QUBITS,
    dtype=np.uint8
)

matches = np.where(
    np.all(ENV_ERRORS == zero_error, axis=1)
)[0]

assert len(matches) == 1

error_index = int(matches[0])

state, info = env.reset(
    error_index=error_index
)

state, reward, terminated, truncated, info = env.step(
    STOP_ACTION
)

print("=" * 70)
print("IMMEDIATE STOP TEST")
print("=" * 70)

print("Correction    :", info["correction"])
print("Residual      :", info["residual"])
print("Logical class :", info["logical_class"])
print("Success       :", info["logical_success"])
print("Reward        :", reward)
print("Terminated    :", terminated)

assert reward == 1.0
assert terminated is True
assert info["logical_success"] is True

print()
print("PASS")

IMMEDIATE STOP TEST
Correction    : [0 0 0 0 0 0 0 0 0]
Residual      : [0 0 0 0 0 0 0 0 0]
Logical class : 0
Success       : True
Reward        : 1.0
Terminated    : True

PASS


In [37]:
# ============================================================
# MAXIMUM STEP TEST
# ============================================================

state, info = env.reset()

terminated = False
truncated = False

for step in range(MAX_STEPS):

    state, reward, terminated, truncated, info = env.step(0)

    if terminated or truncated:
        break

print("=" * 70)
print("MAXIMUM STEP TEST")
print("=" * 70)

print("Steps executed :", env.step_count)
print("Terminated     :", terminated)
print("Truncated      :", truncated)
print("Done           :", env.done)

assert env.step_count == MAX_STEPS
assert truncated is True
assert env.done is True

print()
print("PASS")

MAXIMUM STEP TEST
Steps executed : 9
Terminated     : False
Truncated      : True
Done           : True

PASS


In [38]:
# ============================================================
# ENVIRONMENT UNIT TEST SUMMARY
# ============================================================

print("=" * 70)
print("RL ENVIRONMENT UNIT TESTS")
print("=" * 70)

tests_passed = 0
tests_total = 0


def run_test(name, function):
    global tests_passed, tests_total

    tests_total += 1

    try:
        function()
        print(f"[PASS] {name}")
        tests_passed += 1

    except Exception as exc:
        print(f"[FAIL] {name}")
        print(f"       {type(exc).__name__}: {exc}")


def test_reset():
    state, info = env.reset()

    assert state.shape == (OBSERVATION_DIM,)
    assert state.dtype == np.float32
    assert env.step_count == 0
    assert env.done is False


def test_double_correction():
    env.reset(error_index=0)

    env.step(0)
    env.step(0)

    assert np.array_equal(
        env.correction,
        np.zeros(NUM_DATA_QUBITS, dtype=np.uint8)
    )


def test_success():
    target = np.array(
        [0, 1, 0, 0, 0, 0, 0, 0, 0],
        dtype=np.uint8
    )

    idx = np.where(
        np.all(ENV_ERRORS == target, axis=1)
    )[0][0]

    env.reset(error_index=int(idx))

    env.step(1)

    _, reward, terminated, _, info = env.step(
        STOP_ACTION
    )

    assert reward == 1.0
    assert terminated
    assert info["logical_success"]


def test_failure():
    target = np.array(
        [1, 0, 0, 0, 0, 0, 0, 0, 0],
        dtype=np.uint8
    )

    idx = np.where(
        np.all(ENV_ERRORS == target, axis=1)
    )[0][0]

    env.reset(error_index=int(idx))

    env.step(1)

    _, reward, terminated, _, info = env.step(
        STOP_ACTION
    )

    assert reward == -1.0
    assert terminated
    assert not info["logical_success"]


def test_immediate_stop():
    target = np.zeros(
        NUM_DATA_QUBITS,
        dtype=np.uint8
    )

    idx = np.where(
        np.all(ENV_ERRORS == target, axis=1)
    )[0][0]

    env.reset(error_index=int(idx))

    _, reward, terminated, _, info = env.step(
        STOP_ACTION
    )

    assert reward == 1.0
    assert terminated
    assert info["logical_success"]


def test_max_steps():
    env.reset()

    for _ in range(MAX_STEPS):
        _, _, terminated, truncated, _ = env.step(0)

        if terminated or truncated:
            break

    assert env.step_count == MAX_STEPS
    assert truncated


run_test("Reset", test_reset)
run_test("Double correction cancellation", test_double_correction)
run_test("Successful correction", test_success)
run_test("Failed correction", test_failure)
run_test("Immediate STOP", test_immediate_stop)
run_test("Maximum episode length", test_max_steps)

print()
print("=" * 70)
print(f"RESULT: {tests_passed}/{tests_total} TESTS PASSED")
print("=" * 70)

assert tests_passed == tests_total

RL ENVIRONMENT UNIT TESTS
[PASS] Reset
[PASS] Double correction cancellation
[PASS] Successful correction
[PASS] Failed correction
[PASS] Immediate STOP
[PASS] Maximum episode length

RESULT: 6/6 TESTS PASSED


In [40]:
'''
| Component                   | Status |
| --------------------------- | ------ |
| 512 hidden physical errors  | ✓      |
| 24-bit detector observation | ✓      |
| 9-bit correction frame      | ✓      |
| 10 actions                  | ✓      |
| Sequential correction       | ✓      |
| XOR cancellation            | ✓      |
| Hidden-error isolation      | ✓      |
| Logical reward              | ✓      |
| STOP action                 | ✓      |
| Maximum episode length      | ✓      |
| Successful correction       | ✓      |
| Failed correction           | ✓      |

The observation is:

$$ s_t=[D,C_t]\in\{0,1\}^{33}. $$

The agent does not receive \(E\) or \(L(E)\).

The environment evaluates:
$$ R=E\oplus C_t $$
and gives the terminal reward according to the logical class of \(R\).

A completely random policy that chooses uniformly from all 10 actions can repeatedly toggle corrections and eventually STOP. Because of the symmetry of this controlled dataset, we expect its performance to be around the logical-class baseline, but we should measure it rather than assume it.

We'll run two baselines:

Uniform random policy
Random correction policy with forced STOP

The first tells us how the actual action space behaves.

The second gives a cleaner baseline where the policy randomly chooses corrections but always terminates after a random number of corrections.

'''

"\n| Component                   | Status |\n| --------------------------- | ------ |\n| 512 hidden physical errors  | ✓      |\n| 24-bit detector observation | ✓      |\n| 9-bit correction frame      | ✓      |\n| 10 actions                  | ✓      |\n| Sequential correction       | ✓      |\n| XOR cancellation            | ✓      |\n| Hidden-error isolation      | ✓      |\n| Logical reward              | ✓      |\n| STOP action                 | ✓      |\n| Maximum episode length      | ✓      |\n| Successful correction       | ✓      |\n| Failed correction           | ✓      |\n\nThe observation is:\n\n$$ s_t=[D,C_t]\\in\\{0,1\\}^{33}. $$\n\nThe agent does not receive \\(E\\) or \\(L(E)\\).\n\nThe environment evaluates:\n$$ R=E\\oplus C_t $$\nand gives the terminal reward according to the logical class of \\(R\\).\n\nA completely random policy that chooses uniformly from all 10 actions can repeatedly toggle corrections and eventually STOP. Because of the symmetry of this controll

In [41]:
# ============================================================
# STAGE 6.2 — RANDOM POLICY BASELINE
# ============================================================

def run_random_episode(env):
    """
    Run one episode using a uniformly random policy.

    At every step the policy chooses randomly from all
    available actions, including STOP.
    """

    state, info = env.reset()

    total_reward = 0.0
    actions_taken = []

    while not env.done:

        action = env.rng.integers(
            0,
            env.num_actions
        )

        state, reward, terminated, truncated, info = env.step(
            action
        )

        total_reward += reward
        actions_taken.append(action)

        if terminated or truncated:
            break

    success = bool(
        info.get("logical_success", False)
    )

    return {
        "reward": total_reward,
        "success": success,
        "steps": env.step_count,
        "actions": actions_taken,
        "info": info,
    }

In [42]:
# ============================================================
# RANDOM POLICY — SINGLE EPISODE
# ============================================================

episode = run_random_episode(env)

print("=" * 70)
print("RANDOM POLICY — SINGLE EPISODE")
print("=" * 70)

print("Actions:")
for action in episode["actions"]:
    print(
        f"  {action:2d} -> {ACTION_NAMES[action]}"
    )

print()
print("Steps         :", episode["steps"])
print("Reward        :", episode["reward"])
print("Logical class :", episode["info"]["logical_class"])
print("Success       :", episode["success"])
print("Correction    :", episode["info"]["correction"])
print("Residual      :", episode["info"]["residual"])

RANDOM POLICY — SINGLE EPISODE
Actions:
   8 -> Z_q19
   0 -> Z_q1
   6 -> Z_q15
   2 -> Z_q5
   0 -> Z_q1
   5 -> Z_q12
   9 -> STOP

Steps         : 7
Reward        : 1.0
Logical class : 0
Success       : True
Correction    : [0 0 1 0 0 1 1 0 1]
Residual      : [1 0 0 1 1 1 0 1 1]


In [43]:
# ============================================================
# RANDOM POLICY BASELINE — LARGE EVALUATION
# ============================================================

NUM_RANDOM_EPISODES = 10000

random_rewards = []
random_successes = []
random_steps = []

for _ in range(NUM_RANDOM_EPISODES):

    result = run_random_episode(env)

    random_rewards.append(
        result["reward"]
    )

    random_successes.append(
        result["success"]
    )

    random_steps.append(
        result["steps"]
    )


random_rewards = np.asarray(
    random_rewards,
    dtype=np.float32
)

random_successes = np.asarray(
    random_successes,
    dtype=np.float32
)

random_steps = np.asarray(
    random_steps,
    dtype=np.float32
)


success_rate = 100.0 * random_successes.mean()
mean_reward = random_rewards.mean()
mean_steps = random_steps.mean()


print("=" * 70)
print("RANDOM POLICY BASELINE")
print("=" * 70)

print(f"Episodes             : {NUM_RANDOM_EPISODES}")
print(f"Logical success rate : {success_rate:.2f}%")
print(f"Mean reward          : {mean_reward:.4f}")
print(f"Mean episode length  : {mean_steps:.3f}")

print()
print("Reward distribution:")

unique_rewards, reward_counts = np.unique(
    random_rewards,
    return_counts=True
)

for reward, count in zip(
    unique_rewards,
    reward_counts
):
    print(
        f"  reward={reward:+.1f}: "
        f"{count} "
        f"({100.0 * count / NUM_RANDOM_EPISODES:.2f}%)"
    )

RANDOM POLICY BASELINE
Episodes             : 10000
Logical success rate : 48.90%
Mean reward          : -0.0220
Mean episode length  : 6.142

Reward distribution:
  reward=-1.0: 5110 (51.10%)
  reward=+1.0: 4890 (48.90%)


In [ ]:
'''
A stronger baseline

A random policy isn't the only thing we need.

The current environment contains a very useful piece of information: we know the hidden error internally.

Therefore we can construct an oracle correction policy.

This is not something the RL agent is allowed to use.

It is purely an upper-bound/reference policy.

For any hidden error \(E\), applying the same correction:

$$ C=E $$

produces

$$ E\oplus C=0, $$

so logical success is guaranteed.

This gives us:

Random policy       → lower baseline
RL agent            → learned decoder
Oracle correction   → ideal reference

'''

In [44]:
# ============================================================
# ORACLE BASELINE
# ============================================================

def run_oracle_episode(env, error_index):
    """
    Oracle policy.

    IMPORTANT:
    This policy is NOT available to the RL agent.

    It has access to the hidden physical error and simply
    applies the corresponding correction.

    This establishes an ideal reference point.
    """

    state, info = env.reset(
        error_index=error_index
    )

    hidden_error = env.error.copy()

    correction_actions = np.where(
        hidden_error == 1
    )[0]

    for action in correction_actions:
        state, reward, terminated, truncated, info = env.step(
            int(action)
        )

        assert not terminated
        assert not truncated

    state, reward, terminated, truncated, info = env.step(
        STOP_ACTION
    )

    return {
        "reward": reward,
        "success": info["logical_success"],
        "steps": env.step_count,
        "correction": info["correction"],
        "residual": info["residual"],
        "logical_class": info["logical_class"],
    }

In [45]:
# ============================================================
# EXHAUSTIVE ORACLE VERIFICATION
# ============================================================

oracle_successes = 0
oracle_failures = 0

oracle_steps = []

for error_index in range(len(ENV_ERRORS)):

    result = run_oracle_episode(
        env,
        error_index
    )

    if result["success"]:
        oracle_successes += 1
    else:
        oracle_failures += 1

    oracle_steps.append(
        result["steps"]
    )


oracle_steps = np.asarray(
    oracle_steps,
    dtype=np.int32
)


print("=" * 70)
print("ORACLE BASELINE")
print("=" * 70)

print(f"Errors tested        : {len(ENV_ERRORS)}")
print(f"Successful           : {oracle_successes}")
print(f"Failed               : {oracle_failures}")
print(
    f"Success rate         : "
    f"{100.0 * oracle_successes / len(ENV_ERRORS):.2f}%"
)
print(
    f"Mean episode length  : "
    f"{oracle_steps.mean():.3f}"
)

assert oracle_successes == 512
assert oracle_failures == 0

print()
print("PASS: oracle succeeds on every physical error.")

AssertionError: 